# Layer 8 — Research / Hypothesen Engine

**Rolle:** Die analytische Forschungsebene.
Ebene 8 erhebt oder generiert keine neuen Felddaten. Sie liest das stetig wachsende
Archiv `layer7_history.jsonl` aus und sucht darin nach reproduzierbaren Mustern,
Korrelationen sowie strukturellen Zusammenhängen über verschiedene Snapshots hinweg.

**Architektur:** Tagespaar-orientiert (Vormittag CEST + Abend CEST).
Vormittag = Carnegie-Tiefpunkt (Basislinie), Abend = Carnegie-Höhepunkt (Aktivierung).
ΔL3, ΔL5 und ΔL6 messen die tägliche Aktivierung der jeweiligen Ebene.

**Eingaben:** `layer7_history.jsonl`

**Ausgaben:**
- `layer8_state.json` — maschinenlesbares, vollständiges Analyseergebnis
- `layer8_report.md` — für Menschen lesbarer Zusammenfassungsbericht
- `data/research/hypothesis_registry.json` — persistentes Hypothesenregister
- `data/research/hypothesis_candidates/*.json` — eine Datei pro Hypothesenkandidat

## 0. Setup

In [ ]:
import json, math
from datetime import datetime, timedelta, timezone
from collections import Counter, defaultdict
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Projektpfade (CWD-unabhaengig, ohne pip install) ---
import sys, pathlib
_root = pathlib.Path.cwd().resolve()
while not (_root / '.project-root').exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root / 'src'))
from atmosphere.paths import layer_state, HISTORY, REPORTS, STATES_REGISTRY

HISTORY_FILE = HISTORY
STATE_FILE   = layer_state(8)
REPORT_FILE  = REPORTS / 'layer8_report.md'

from zoneinfo import ZoneInfo
LOCAL_TZ = ZoneInfo('Europe/Berlin')   # nur Anzeige; die Logik laeuft auf UTC

# Slot-Grenzen in UTC-Stunden. Bewusst an den L4-Tag/Nacht-Schaltpunkten
# (6 und 19 UTC) ausgerichtet, damit jeder Slot cavity-homogen ist:
# night/evening liegen im Nacht-Regime (h~87 km), morning/midday im Tag (h~70 km).
SLOT_BOUNDS_UTC = [
    ('night',    0,  6),   # 00:00-05:59 UTC
    ('morning',  6, 12),   # 06:00-11:59 UTC
    ('midday',  12, 19),   # 12:00-18:59 UTC
    ('evening', 19, 24),   # 19:00-23:59 UTC
]
SLOT_ORDER = [s[0] for s in SLOT_BOUNDS_UTC]

# Primaeres Tagespaar: beide Slots im Nacht-Regime, nahe Carnegie-Trog bzw.
# -Peak. Der 17-km-Cavity-Sprung kuerzt sich heraus statt als Signal aufzutreten.
PRIMARY_PAIR = ('night', 'evening')

def slot_of(t_utc):
    """UTC-Zeitstempel -> Slot-Name."""
    h = t_utc.hour
    for name, lo, hi in SLOT_BOUNDS_UTC:
        if lo <= h < hi:
            return name
    return 'unknown'
RUN_TS        = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S.%fZ')
ENGINE_VERSION = '1.0'

def evidence_level(n):
    if n is None:   return 'unknown'
    if n < 10:      return 'exploratory_signal'
    if n < 30:      return 'testable'
    if n < 100:     return 'moderate_evidence'
    return 'robust_candidate'

print(f'Layer 8 Engine {ENGINE_VERSION}  |  Run: {RUN_TS}')

## 1. History laden & deduplizieren

Doppelte Snapshots (gleicher Timestamp) entstehen durch Re-Runs derselben Layer-7-Session. Wir behalten nur den ersten.

In [ ]:
# KANONISCHE Stichprobe aus dem gemeinsamen Loader — dieselbe Definition, die
# L9, holarchic und der Analyse-Layer verwenden. Vorher dedupte diese Zelle
# selbst mit keep-FIRST und verwarf damit bei jedem Duplikat den REICHEREN
# Datensatz (die spaeteren Wiederholungslaeufe tragen field_operators /
# field_operator_vector, die frueheren nicht) — genau die Felder, aus denen die
# Lead-Lag-Analyse ihre Werte zieht. Der Loader nutzt keep-LAST.
from atmosphere.history import load_history, sample_signature, schema_coverage

snaps, hist_meta = load_history(HISTORY_FILE)
history = snaps          # Rueckwaertskompatibilitaet fuer nachfolgende Zellen

# Slot-Annotation (Logik auf UTC, Anzeige lokal)
for s in snaps:
    t_utc = datetime.fromisoformat(s['timestamp'].replace('Z','')).replace(tzinfo=timezone.utc)
    s['_utc']   = t_utc
    s['_date']  = t_utc.strftime('%Y-%m-%d')     # UTC-Datum: alle 4 Slots fallen hinein
    s['_slot']  = slot_of(t_utc)
    s['_mesz']  = t_utc.astimezone(LOCAL_TZ)     # nur fuer die Ausgabe

print(f'Stichprobe:               {sample_signature(hist_meta)}')
print(f'Engines:                  {hist_meta["engine_versions"]}')
_cov = schema_coverage(snaps)
_partial = {f: d['pct'] for f, d in _cov.items() if f != '_layers' and d['pct'] < 100}
if _partial:
    print(f'Teil-Deckung (Schema gewachsen): {_partial}')
print(f'Zeitraum:                 {snaps[0]["_mesz"].strftime("%Y-%m-%d %H:%M")} – {snaps[-1]["_mesz"].strftime("%Y-%m-%d %H:%M")} MESZ')
_slot_counts = Counter(s['_slot'] for s in snaps)
print('Snapshots je Slot (UTC-Stunden):')
for _name in SLOT_ORDER + (['unknown'] if _slot_counts.get('unknown') else []):
    print(f'  {_name:<9} {_slot_counts.get(_name, 0):>4}')

## 2. Tagespaare bilden

Jeder Tag soll ein Paar (Morning + Evening) haben. Tagespaare sind die zentrale Analyseeinheit, weil ΔL3, ΔL5 und ΔOperator nur dann sinnvoll sind.

In [ ]:
by_date = defaultdict(dict)
for s in snaps:
    # bei mehreren Snapshots im selben Slot: ersten behalten (stabile Baseline)
    if s['_slot'] not in by_date[s['_date']]:
        by_date[s['_date']][s['_slot']] = s

day_pairs = []
for date in sorted(by_date.keys()):
    slots = by_date[date]
    a, b  = PRIMARY_PAIR
    day_pairs.append({
        'date':     date,
        'slots':    slots,                 # alle vorhandenen Slots des Tages
        'n_slots':  len(slots),
        'morning':  slots.get(a),          # rueckwaertskompatibel: primaeres Paar
        'evening':  slots.get(b),          # (Zellen 10/12/14 lesen diese Keys)
        'complete': bool(slots.get(a) and slots.get(b)),
        'full_day': len(slots) == len(SLOT_ORDER),
    })

complete_pairs = [d for d in day_pairs if d['complete']]
n_pairs        = len(complete_pairs)
n_full_days    = sum(1 for d in day_pairs if d['full_day'])

print(f'Tage gesamt:              {len(day_pairs)}')
print(f'Primaerpaare {PRIMARY_PAIR[0]}<->{PRIMARY_PAIR[1]}: {n_pairs}')
print(f'Tage mit allen 4 Slots:   {n_full_days}')
print()
for d in day_pairs:
    icon  = '\u2713' if d['complete'] else '\u2717'
    parts = [f'{k}={d["slots"][k]["_mesz"].strftime("%H:%M")}'
             if k in d['slots'] else f'{k}=\u2014' for k in SLOT_ORDER]
    print(f'  {icon} {d["date"]}  {" ".join(parts)}')


## 3. State- und Layer-Profil

Häufigkeit der System-States, Bandbreite der Layer-Scores, dominante Layer.

In [ ]:
LAYERS = ['L0_external_drivers', 'L1_planetary_body', 'L2_surface_zone',
          'L3_atmosphere', 'L4_ionosphere', 'L5_global_electric_circuit',
          'L6_resonance_field']

def lscore(snap, lname):
    return snap['layers'].get(lname, {}).get('score')

# State-Häufigkeit
state_counts = Counter(s['system_state'] for s in snaps)
state_freq = {state: {'count': c, 'pct': round(c/len(snaps)*100, 1)}
              for state, c in state_counts.most_common()}

# Layer-Statistik
layer_stats = {}
for lname in LAYERS:
    vals = [lscore(s, lname) for s in snaps]
    vals = [v for v in vals if v is not None]
    if vals:
        layer_stats[lname] = {
            'mean':       round(float(np.mean(vals)),  4),
            'std':        round(float(np.std(vals)),   4),
            'min':        round(float(np.min(vals)),   4),
            'max':        round(float(np.max(vals)),   4),
            'range':      round(float(np.max(vals) - np.min(vals)), 4),
        }

# Dominanz-Häufigkeit
dom_counts = Counter(s['dominance']['dominant_layer'] for s in snaps if s.get('dominance'))

# Ausgabe
print('SYSTEM STATES')
print('=' * 65)
for state, stats in state_freq.items():
    bar = '█' * stats['count']
    print(f'  {state:<38} {bar} {stats["count"]:>2}× ({stats["pct"]:.0f}%)')

print()
print('LAYER SCORE PROFIL')
print('=' * 65)
print(f'  {"Layer":<32} {"mean":>6} {"std":>6} {"min":>6} {"max":>6}')
for lname, st in layer_stats.items():
    print(f'  {lname:<32} {st["mean"]:>6.3f} {st["std"]:>6.3f} {st["min"]:>6.3f} {st["max"]:>6.3f}')

print()
print('DOMINANZ')
print('=' * 65)
for layer, c in dom_counts.most_common():
    print(f'  {layer:<32} {c:>2}× ({c/len(snaps)*100:.0f}%)')

## 4. Morning vs Evening — ΔL3 als Aktivierungsindikator

ΔL3 = L3-Score abends − L3-Score morgens.
ΔL3 zeigt wie stark sich die Atmosphäre tagsüber aktiviert hat.

**Hypothese:** ΔL3 > ~0.035 ist Voraussetzung für `anomalous_resonance_state`.

In [ ]:
dl3_data = []
for d in complete_pairs:
    m, e = d['morning'], d['evening']
    dl3 = lscore(e, 'L3_atmosphere') - lscore(m, 'L3_atmosphere')
    dl5 = lscore(e, 'L5_global_electric_circuit') - lscore(m, 'L5_global_electric_circuit')
    dl6 = lscore(e, 'L6_resonance_field')         - lscore(m, 'L6_resonance_field')
    dl3_data.append({
        'date':           d['date'],
        'L3_morning':     round(lscore(m, 'L3_atmosphere'), 4),
        'L3_evening':     round(lscore(e, 'L3_atmosphere'), 4),
        'delta_L3':       round(dl3, 4),
        'delta_L5':       round(dl5, 4),
        'delta_L6':       round(dl6, 4),
        'evening_state':  e['system_state'],
        'evening_L5':     round(lscore(e, 'L5_global_electric_circuit'), 4),
    })

# ΔL3 Schwellenwert empirisch suchen
anomal_dl3   = [r['delta_L3'] for r in dl3_data if r['evening_state'] == 'anomalous_resonance_state']
seasonal_dl3 = [r['delta_L3'] for r in dl3_data if r['evening_state'] != 'anomalous_resonance_state']

dl3_threshold = None
if anomal_dl3 and seasonal_dl3:
    # niedrigster anomalous Wert minus 0.005 als grobe Schwelle
    dl3_threshold = round(min(anomal_dl3) - 0.005, 4)

# ============================================================
# COMBINED ACTIVATION SCORE — besser als ΔL3 allein?
# combined = 0.4*ΔL3_norm + 0.3*L5_evening + 0.3*L6_evening
# ============================================================

combined_data = []
for r in dl3_data:
    # ΔL3 normieren: typischer Range ~[-0.2, +0.3] → auf [0,1]
    dl3_norm = min(1.0, max(0.0, (r['delta_L3'] + 0.2) / 0.5))
    l5_eve   = r['evening_L5']
    l6_eve   = lscore(by_date[r['date']]['evening'], 'L6_resonance_field') or 0

    combined = round(0.4 * dl3_norm + 0.3 * l5_eve + 0.3 * l6_eve, 4)
    is_anomal = r['evening_state'] == 'anomalous_resonance_state'

    combined_data.append({
        'date':       r['date'],
        'delta_L3':   r['delta_L3'],
        'dl3_norm':   round(dl3_norm, 4),
        'L5_evening': round(l5_eve, 4),
        'L6_evening': round(l6_eve, 4),
        'combined':   combined,
        'anomalous':  is_anomal,
    })

anomal_combined   = [d['combined'] for d in combined_data if d['anomalous']]
seasonal_combined = [d['combined'] for d in combined_data if not d['anomalous']]

combined_threshold = None
if anomal_combined and seasonal_combined:
    combined_threshold = round(min(anomal_combined) - 0.01, 4)
    overlap_combined   = max(seasonal_combined) >= min(anomal_combined)
    overlap_dl3        = bool(seasonal_dl3 and anomal_dl3 and
                              max(seasonal_dl3) >= min(anomal_dl3))

print('COMBINED ACTIVATION SCORE')
print('=' * 78)
print(f'  {"Datum":<10} {"ΔL3":>7} {"L5eve":>7} {"L6eve":>7} {"combined":>9}  State')
for d in combined_data:
    state = '⚡ ANOMAL' if d['anomalous'] else '  seasonal'
    print(f'  {d["date"]}  {d["delta_L3"]:>+7.3f} {d["L5_evening"]:>7.3f} '
          f'{d["L6_evening"]:>7.3f} {d["combined"]:>9.4f}  {state}')

if combined_threshold is not None:
    print(f'\n  Combined bei anomalous:  mean={np.mean(anomal_combined):.3f}  min={min(anomal_combined):.3f}')
    print(f'  Combined bei seasonal:   mean={np.mean(seasonal_combined):.3f}  max={max(seasonal_combined):.3f}')
    print(f'  Schwelle combined:       {combined_threshold:.4f}  (Overlap: {overlap_combined})')
    print(f'  Schwelle ΔL3 allein:     {dl3_threshold}  (Overlap: {overlap_dl3})')
    better = 'combined besser' if not overlap_combined and overlap_dl3 else \
             'ΔL3 besser' if overlap_combined and not overlap_dl3 else \
             'beide gleich gut' if not overlap_combined and not overlap_dl3 else \
             'beide noch unsicher'
    print(f'  → {better}')

## 5. Carnegie Amplitude — was bestimmt L5 abends?

Der Carnegie-Tagesgang ist real (alle 4 anomalous Events 17–20 UTC), aber die **Amplitude** schwankt von Tag zu Tag. Was korreliert mit hohem L5-Abendwert?

In [ ]:
# Korrelationen mit L5_evening über Abend-Snapshots
evening_snaps = [d['evening'] for d in complete_pairs]

if len(evening_snaps) >= 4:
    l5_eve = np.array([lscore(s, 'L5_global_electric_circuit') for s in evening_snaps])

    candidates = {
        'L3_evening':    np.array([lscore(s, 'L3_atmosphere') for s in evening_snaps]),
        'L2_evening':    np.array([lscore(s, 'L2_surface_zone') for s in evening_snaps]),
        'L6_evening':    np.array([lscore(s, 'L6_resonance_field') for s in evening_snaps]),
        'L0_evening':    np.array([lscore(s, 'L0_external_drivers') for s in evening_snaps]),
        'delta_L3':      np.array([d['evening']['layers']['L3_atmosphere']['score']
                                   - d['morning']['layers']['L3_atmosphere']['score']
                                   for d in complete_pairs]),
        'L2_morning':    np.array([lscore(d['morning'], 'L2_surface_zone') for d in complete_pairs]),
    }

    correlations = {}
    for name, arr in candidates.items():
        if arr.std() > 0 and l5_eve.std() > 0:
            r = float(np.corrcoef(l5_eve, arr)[0, 1])
            correlations[name] = round(r, 4)

    correlations_sorted = sorted(correlations.items(), key=lambda x: -abs(x[1]))

    print('CARNEGIE AMPLITUDE — Korrelationen mit L5_evening')
    print('=' * 65)
    print(f'  {"Variable":<20} {"Pearson r":>11}   Interpretation')
    for name, r in correlations_sorted:
        if abs(r) > 0.7:    interp = 'starker Zusammenhang'
        elif abs(r) > 0.4:  interp = 'mittlerer Zusammenhang'
        elif abs(r) > 0.2:  interp = 'schwacher Zusammenhang'
        else:               interp = 'kein Zusammenhang'
        sign = '+' if r >= 0 else '−'
        print(f'  {name:<20} {sign}{abs(r):>10.3f}   {interp}')

    carnegie_amplitude = {
        'n_evenings':       len(evening_snaps),
        'L5_evening_mean':  round(float(l5_eve.mean()), 4),
        'L5_evening_std':   round(float(l5_eve.std()),  4),
        'correlations':     correlations,
        'top_predictor':    correlations_sorted[0][0] if correlations_sorted else None,
    }
else:
    print(f'Nur {len(evening_snaps)} Abend-Snapshots — Korrelation nicht aussagekräftig.')
    print('Ab >= 4 Abend-Snapshots aktiv.')
    carnegie_amplitude = {'n_evenings': len(evening_snaps), 'note': 'insufficient_data'}

## 6. L2→L3 Aktivierungsparadox

L2 (ENSO/SST) ist dauerhaft hoch, aber L3 (Atmosphäre) aktiviert sich nicht entsprechend.
Wir prüfen ob L2 und L3 auf gleicher Zeitskala leben oder L2 ein "Slow Driver" ist.

In [ ]:
l2_all  = np.array([lscore(s, 'L2_surface_zone') for s in snaps])
l3_all  = np.array([lscore(s, 'L3_atmosphere')   for s in snaps])
gap_all = l2_all - l3_all

pearson_l2_l3 = float(np.corrcoef(l2_all, l3_all)[0, 1]) if l2_all.std() > 0 else 0.0

# Trend pro Layer (linearer Fit über Index = Zeit)
def linear_trend(arr):
    if len(arr) < 3 or np.std(arr) == 0:
        return 0.0
    x = np.arange(len(arr))
    slope = float(np.polyfit(x, arr, 1)[0])
    return round(slope, 5)

l2_trend  = linear_trend(l2_all)
l3_trend  = linear_trend(l3_all)
gap_trend = linear_trend(gap_all)

print('L2 ↔ L3 BEZIEHUNG')
print('=' * 65)
print(f'  Pearson L2 vs L3:        {pearson_l2_l3:+.4f}')
print(f'  L2 Trend (Δ/Snapshot):   {l2_trend:+.5f}')
print(f'  L3 Trend (Δ/Snapshot):   {l3_trend:+.5f}')
print(f'  Gap Trend (Δ/Snapshot):  {gap_trend:+.5f}')
print()
print(f'  Gap-Mittel:              {float(gap_all.mean()):.3f}')
print(f'  Gap-Range:               {float(gap_all.min()):.3f} – {float(gap_all.max()):.3f}')

if pearson_l2_l3 < -0.2:
    interp = 'NEGATIVE Korrelation: L2 (Wochen-Trend) und L3 (Tagesrhythmus) leben auf verschiedenen Zeitskalen.'
elif pearson_l2_l3 > 0.5:
    interp = 'POSITIV gekoppelt: L2 und L3 bewegen sich synchron.'
else:
    interp = 'Schwache Kopplung sichtbar — mehr Daten nötig.'

print(f'\n  → {interp}')

l2_l3_paradox = {
    'pearson':       round(pearson_l2_l3, 4),
    'l2_trend':      l2_trend,
    'l3_trend':      l3_trend,
    'gap_trend':     gap_trend,
    'gap_mean':      round(float(gap_all.mean()), 4),
    'gap_max':       round(float(gap_all.max()),  4),
    'interpretation': interp,
}

## 7. Coupling Analysis

Welche Kopplungen sind stabil, welche schwanken? Die Variabilität ist oft aussagekräftiger als der Mittelwert.

In [ ]:
coupling_stats = {}
for s in snaps:
    for c in s.get('couplings', []):
        key = f'{c["from"].split("_")[0]}→{c["to"].split("_")[0]}'
        coupling_stats.setdefault(key, []).append(c['strength'])

coupling_summary = {}
for key, vals in coupling_stats.items():
    coupling_summary[key] = {
        'mean':       round(float(np.mean(vals)), 4),
        'std':        round(float(np.std(vals)),  4),
        'min':        round(float(np.min(vals)),  4),
        'max':        round(float(np.max(vals)),  4),
        'volatility': round(float(np.std(vals) / np.mean(vals)) if np.mean(vals) > 0 else 0, 4),
    }

print('KOPPLUNGS-PROFIL  (sortiert nach Volatilität)')
print('=' * 78)
print(f'  {"Kopplung":<10} {"mean":>7} {"std":>7} {"min":>7} {"max":>7} {"CV":>7}  Interpretation')
for key, st in sorted(coupling_summary.items(), key=lambda x: -x[1]['volatility']):
    if st['volatility'] > 0.5:    note = 'sehr variabel — situations-getrieben'
    elif st['volatility'] > 0.2:  note = 'moderat variabel'
    else:                         note = 'stabil — strukturell'
    print(f'  {key:<10} {st["mean"]:>7.3f} {st["std"]:>7.3f} {st["min"]:>7.3f} {st["max"]:>7.3f} {st["volatility"]:>7.3f}  {note}')

## 8. Field Operators — Wirkkraft-Analyse

Die Field Operators beschreiben die *Wirkkräfte* hinter den Layern. Diese Sektion ist erst voll aktiv, sobald mehrere Snapshots Operatoren enthalten (Layer 7 wurde später erweitert).

**Was hier passiert:**
1. Operator-Statistik über alle verfügbaren Snapshots
2. Welcher Operator dominiert bei welchem System-State?
3. Korrelation Operator vs L3-Aktivierung (ΔL3)
4. Ranking der aktuellen Operatoren

In [ ]:
# ============================================================
# 8. FIELD OPERATORS — Wirkkraft-Analyse
# ============================================================
# Layer 8 analysiert Operatoren als Zeitreihen und Musterindikatoren:
# 8.1 Coverage + Regime-Warnung
# 8.2 Statistik + Trend (current, previous, delta, rolling)
# 8.3 Lead-Lag: Operator(t) → Layer(t+k)
# 8.4 Operator-Regime (Profil-Klassifikation)
# 8.5 State Precursors (Operatoren bei t-1)
# 8.6 Operator-Kombinationen (gleichzeitig hoch)
# 8.7 Operator vs ΔL3 (für Hypothesen-Tracker)
# 8.8 Aktuelle Rangliste
# ============================================================

from itertools import combinations

# Operator-Namen (mit Rückwärtskompatibilität für alte Snapshots)
OP_NAMES = ['thermal_operator', 'electric_operator', 'ionization_operator',
            'geomagnetic_operator', 'resonance_model_operator',
            'tidal_gravity_operator', 'cross_layer_activation_operator']
OP_ALIASES = {'resonance_operator': 'resonance_model_operator'}

HIGH = 0.5   # Schwelle "hoch"
LOW  = 0.3   # Schwelle "niedrig"

def _get_op_full(snap, op_name):
    """Holt Operator-Objekt, behandelt Aliasse"""
    ops = snap.get('field_operators') or {}
    o = ops.get(op_name)
    if o is None:
        for old, new in OP_ALIASES.items():
            if new == op_name and old in ops:
                return ops[old]
    return o

def _get_op(snap, op_name):
    """Holt nur den Score, oder None"""
    o = _get_op_full(snap, op_name)
    if o and isinstance(o, dict) and o.get('score') is not None:
        return float(o['score'])
    return None

ops_snaps = [s for s in snaps if s.get('field_operators')]
ops_count = len(ops_snaps)
coverage_pct = round(ops_count / len(snaps) * 100, 1) if snaps else 0

# ── 8.1 Coverage + Regime ────────────────────────────────────
if ops_count < 30:
    coverage_regime = 'exploratory'
    coverage_note   = 'exploratory only — operator coverage zu gering für robuste Muster'
elif ops_count < 100:
    coverage_regime = 'first_patterns'
    coverage_note   = 'erste Muster sichtbar — vorsichtig interpretieren'
elif ops_count < 300:
    coverage_regime = 'transition_analysis'
    coverage_note   = 'brauchbare Übergangsanalyse möglich'
else:
    coverage_regime = 'robust'
    coverage_note   = 'robuste saisonale/tageszeitliche Analyse möglich'

operator_analysis = {
    'snapshots_with_operators': ops_count,
    'total_snapshots':          len(snaps),
    'coverage_pct':             coverage_pct,
    'coverage_regime':          coverage_regime,
    'coverage_note':            coverage_note,
}

print('FIELD OPERATORS')
print('=' * 78)
print(f'  Coverage:  {ops_count}/{len(snaps)} ({coverage_pct}%)')
print(f'  Regime:    {coverage_regime}')
print(f'  → {coverage_note}')

if ops_count == 0:
    print('\n  ❌ Keine Operator-Daten. Layer-7-Update nötig.')
    operator_analysis['status'] = 'no_data'

else:
    # ── 8.2 Statistik + Trend ────────────────────────────────
    op_stats  = {}
    op_trends = {}
    op_series_cache = {}   # für 8.3 wiederverwenden

    for op in OP_NAMES:
        vals = [_get_op(s, op) for s in ops_snaps]
        op_series_cache[op] = vals
        vals_clean = [v for v in vals if v is not None]

        if not vals_clean:
            op_stats[op] = None
            op_trends[op] = None
            continue

        op_stats[op] = {
            'count': len(vals_clean),
            'mean':  round(float(np.mean(vals_clean)), 4),
            'std':   round(float(np.std(vals_clean)),  4),
            'min':   round(float(np.min(vals_clean)),  4),
            'max':   round(float(np.max(vals_clean)),  4),
        }

        current  = vals_clean[-1]
        previous = vals_clean[-2] if len(vals_clean) >= 2 else None
        delta    = round(current - previous, 4) if previous is not None else None
        # Rolling: 1d ≈ 4 Snapshots, 3d ≈ 12 Snapshots
        roll_1d  = round(float(np.mean(vals_clean[-4:])),  4) if len(vals_clean) >= 2 else None
        roll_3d  = round(float(np.mean(vals_clean[-12:])), 4) if len(vals_clean) >= 4 else None

        if delta is None:        direction = 'unknown'
        elif delta > 0.03:       direction = 'rising'
        elif delta < -0.03:      direction = 'falling'
        else:                    direction = 'stable'

        op_trends[op] = {
            'current':    round(current, 4),
            'previous':   round(previous, 4) if previous is not None else None,
            'delta':      delta,
            'rolling_1d': roll_1d,
            'rolling_3d': roll_3d,
            'direction':  direction,
        }

    operator_analysis['operator_stats']  = op_stats
    operator_analysis['operator_trends'] = op_trends

    print('\n  ── 8.2 Statistik + Trend ──')
    print(f'  {"Operator":<32} {"curr":>6} {"prev":>6} {"Δ":>7} {"1d":>6} {"3d":>6}  dir')
    for op in OP_NAMES:
        t = op_trends.get(op)
        if not t: continue
        short = op.replace('_operator', '')
        prev  = f'{t["previous"]:.3f}'   if t['previous']   is not None else '   –'
        delt  = f'{t["delta"]:+.3f}'     if t['delta']      is not None else '   –'
        r1    = f'{t["rolling_1d"]:.3f}' if t['rolling_1d'] is not None else '   –'
        r3    = f'{t["rolling_3d"]:.3f}' if t['rolling_3d'] is not None else '   –'
        arrow = {'rising':'↑','falling':'↓','stable':'→','unknown':'?'}[t['direction']]
        print(f'  {short:<32} {t["current"]:>6.3f} {prev:>6} {delt:>7} {r1:>6} {r3:>6}  {arrow} {t["direction"]}')

    # ── 8.3 Lead-Lag-Analyse ─────────────────────────────────
    LAYERS_TO_PREDICT = ['L3_atmosphere', 'L4_ionosphere',
                         'L5_global_electric_circuit', 'L6_resonance_field']
    LAGS = [1, 2, 4]   # ~6h, ~12h, ~24h bei 4 Snapshots/Tag

    lead_lag = {}

    for op in OP_NAMES:
        if not op_stats.get(op): continue
        op_arr = np.array([v if v is not None else np.nan for v in op_series_cache[op]])

        for layer_target in LAYERS_TO_PREDICT:
            tgt = np.array([s['layers'].get(layer_target, {}).get('score')
                            if s['layers'].get(layer_target, {}).get('score') is not None
                            else np.nan for s in ops_snaps])

            for lag in LAGS:
                if len(ops_snaps) <= lag + 3:
                    continue
                a = op_arr[:-lag]
                b = tgt[lag:]
                mask = ~(np.isnan(a) | np.isnan(b))
                if mask.sum() < 4: continue
                a2, b2 = a[mask], b[mask]
                if a2.std() == 0 or b2.std() == 0: continue
                r = float(np.corrcoef(a2, b2)[0, 1])
                if abs(r) > 0.3:
                    key = f'{op.replace("_operator","")}_t__{layer_target}_t+{lag}'
                    lead_lag[key] = {
                        'pearson':       round(r, 4),
                        'n_pairs':       int(mask.sum()),
                        'lag_snapshots': lag,
                    }

    operator_analysis['lead_lag'] = lead_lag

    if lead_lag:
        print('\n  ── 8.3 Lead-Lag: Operator(t) → Layer(t+k)   (|r| > 0.3) ──')
        for key, info in sorted(lead_lag.items(), key=lambda x: -abs(x[1]['pearson'])):
            r = info['pearson']
            strength = 'stark' if abs(r) > 0.7 else 'mittel' if abs(r) > 0.5 else 'schwach'
            print(f'  {key:<52} r = {r:+.3f}  (n={info["n_pairs"]}, {strength})')
    else:
        print(f'\n  ── 8.3 Lead-Lag ──   keine Korrelation mit |r| > 0.3 (n_ops={ops_count})')

    # ── 8.4 Operator-Regime (perzentil-basiert) ──────────────
    # Globale HIGH=0.5 trifft kaum, weil die Operatoren je eigenen
    # Wertebereich haben (thermal ~0.4, geomagnetic ~0.17). Daher:
    # pro Operator das obere/untere Drittel der eigenen Verteilung.
    op_percentiles = {}
    for op in OP_NAMES:
        vals = [_get_op(s, op) for s in ops_snaps]
        vals = [v for v in vals if v is not None]
        if len(vals) >= 6:
            op_percentiles[op] = {
                'p33': float(np.percentile(vals, 33)),
                'p67': float(np.percentile(vals, 67)),
            }

    def op_level(snap, op):
        v = _get_op(snap, op)
        p = op_percentiles.get(op)
        if v is None or p is None:
            return None
        if v >= p['p67']: return 'high'
        if v <  p['p33']: return 'low'
        return 'mid'

    def classify_regime(snap):
        levels = {op: op_level(snap, op) for op in OP_NAMES}
        def hi(op): return levels.get(op) == 'high'
        def lo(op): return levels.get(op) == 'low'

        # Reihenfolge: spezifisch → allgemein
        if hi('thermal_operator') and hi('electric_operator') and hi('resonance_model_operator'):
            return 'thermal_electric_resonance_coupled'
        if hi('thermal_operator') and hi('electric_operator'):
            return 'thermal_electric_coupled'
        if hi('cross_layer_activation_operator') and lo('electric_operator'):
            return 'surface_prepared_delayed'
        if hi('electric_operator') and lo('resonance_model_operator'):
            return 'electric_without_resonance'
        if hi('resonance_model_operator') and lo('electric_operator'):
            return 'resonance_residual'
        if hi('ionization_operator') or hi('geomagnetic_operator'):
            return 'space_weather_mode'
        if hi('thermal_operator') and lo('electric_operator') and lo('resonance_model_operator'):
            return 'thermal_prepared'
        if all(lo(op) for op in OP_NAMES if levels.get(op) is not None):
            return 'quiet_background'
        return 'mixed'

    regime_assignments = []
    for s in ops_snaps:
        regime_assignments.append({
            'date':   s['_date'],
            'slot':   s['_slot'],
            'state':  s['system_state'],
            'regime': classify_regime(s),
        })
    regime_counts = Counter(r['regime'] for r in regime_assignments)

    operator_analysis['operator_regimes'] = {
        'counts':      dict(regime_counts),
        'thresholds':  {op: {'p33': round(p['p33'], 3), 'p67': round(p['p67'], 3)}
                        for op, p in op_percentiles.items()},
        'assignments': regime_assignments,
    }

    print('\n  ── 8.4 Operator-Regime ──')
    for regime, c in regime_counts.most_common():
        pct = round(c / ops_count * 100, 1)
        bar = '█' * c
        print(f'  {regime:<28} {bar} {c}× ({pct}%)')

    # ── 8.5 State Precursors (Operatoren bei t-1) ────────────
    state_precursors = defaultdict(lambda: defaultdict(list))
    for i, s in enumerate(ops_snaps):
        if i == 0: continue
        prev_snap = ops_snaps[i-1]
        cur_state = s['system_state']
        for op in OP_NAMES:
            v = _get_op(prev_snap, op)
            if v is not None:
                state_precursors[cur_state][op].append(v)

    precursor_summary = {}
    for state, ops_dict in state_precursors.items():
        per_op = {op: {'mean_t_minus_1': round(float(np.mean(vals)), 4),
                        'n':              len(vals)}
                   for op, vals in ops_dict.items() if vals}
        top = sorted(per_op.items(), key=lambda x: -x[1]['mean_t_minus_1'])[:3]
        precursor_summary[state] = {
            'all_operators': per_op,
            'top_3':         [{'op': k, **v} for k, v in top],
        }

    operator_analysis['state_precursors'] = precursor_summary

    print('\n  ── 8.5 State Precursors (Operatoren bei t-1) ──')
    for state, info in precursor_summary.items():
        top_str = ', '.join(f'{x["op"].replace("_operator","")}={x["mean_t_minus_1"]:.2f}'
                              for x in info['top_3'])
        print(f'  before {state.replace("_state",""):<28} → {top_str}')

    # ── 8.6 Operator-Kombinationen (perzentil-basiert) ───────
    # Statt globaler Schwelle HIGH=0.5: nutze 'high'-Level pro Operator
    # (oberes Drittel der eigenen Verteilung, schon in 8.4 berechnet).
    combo_counts    = Counter()
    combo_followups = defaultdict(list)

    for i, s in enumerate(ops_snaps):
        high_ops = [op.replace('_operator','') for op in OP_NAMES
                    if op_level(s, op) == 'high']
        for size in (2, 3):
            for combo in combinations(sorted(high_ops), size):
                combo_counts[combo] += 1
                if size == 2 and i + 1 < len(ops_snaps):
                    cur_l3 = s['layers'].get('L3_atmosphere', {}).get('score')
                    nxt_l3 = ops_snaps[i+1]['layers'].get('L3_atmosphere', {}).get('score')
                    if cur_l3 is not None and nxt_l3 is not None:
                        combo_followups[combo].append(nxt_l3 - cur_l3)

    combo_analysis = {}
    for combo, count in combo_counts.most_common(10):
        key = ' + '.join(combo)
        entry = {'count': count}
        if combo in combo_followups and combo_followups[combo]:
            deltas = combo_followups[combo]
            entry['mean_l3_delta_next'] = round(float(np.mean(deltas)), 4)
            entry['pct_l3_rising_next'] = round(
                sum(1 for d in deltas if d > 0.03) / len(deltas) * 100, 1)
        combo_analysis[key] = entry

    operator_analysis['operator_combinations'] = combo_analysis

    if combo_analysis:
        print('\n  ── 8.6 Operator-Kombinationen (gleichzeitig hoch) ──')
        for key, info in list(combo_analysis.items())[:8]:
            extra = ''
            if 'mean_l3_delta_next' in info:
                extra = f'  → ΔL3 next: {info["mean_l3_delta_next"]:+.3f} ({info["pct_l3_rising_next"]:.0f}% rising)'
            print(f'  {key:<48} {info["count"]}×{extra}')

    # ── 8.7 Operator vs ΔL3  (für Hypothesen-Tracker) ────────
    pairs_with_ops = []
    for d in complete_pairs:
        e = d['evening']
        if not e.get('field_operators'): continue
        dl3 = lscore(e, 'L3_atmosphere') - lscore(d['morning'], 'L3_atmosphere')
        pairs_with_ops.append((dl3, e))

    op_dl3_corr = {}
    if len(pairs_with_ops) >= 4:
        dl3_arr = np.array([p[0] for p in pairs_with_ops])
        for op in OP_NAMES:
            vals = [_get_op(snap, op) for _, snap in pairs_with_ops]
            if all(v is not None for v in vals):
                arr = np.array(vals)
                if arr.std() > 0 and dl3_arr.std() > 0:
                    op_dl3_corr[op] = round(float(np.corrcoef(arr, dl3_arr)[0, 1]), 4)
        if op_dl3_corr:
            print('\n  ── 8.7 Operator vs ΔL3 (Aktivierung, abends) ──')
            for op, r in sorted(op_dl3_corr.items(), key=lambda x: -abs(x[1])):
                print(f'  {op.replace("_operator",""):<32} r = {r:+.3f}')
    else:
        print(f'\n  ── 8.7 Operator vs ΔL3 ──   aktiviert ab >= 4 Tagespaaren mit Operatoren')

    operator_analysis['operator_dl3_correlation'] = op_dl3_corr

    # ── 8.8 Aktuelle Rangliste ───────────────────────────────
    latest = ops_snaps[-1]
    ranked = []
    for name in OP_NAMES:
        v = _get_op(latest, name)
        if v is None: continue
        o = _get_op_full(latest, name)
        interp = o.get('interpretation', '') if isinstance(o, dict) else ''
        ranked.append((name, v, interp))
    ranked.sort(key=lambda x: -x[1])

    print('\n  ── 8.8 Aktuelle Operator-Rangliste ──')
    for name, score, interp in ranked:
        short = name.replace('_operator', '')
        bar = '▰' * int(score * 10) + '▱' * (10 - int(score * 10))
        print(f'  {short:<32} {bar} {score:.3f}')
        if interp:
            print(f'    └─ {interp[:80]}')

    operator_analysis['latest_ranking'] = [
        {'operator': n, 'score': s, 'interpretation': i} for n, s, i in ranked
    ]
    operator_analysis['status'] = 'active'

In [ ]:
# ============================================================
# 9. ENSO COUPLING TRACKER
# Analysiert ENSO-Klassen als Systemtreiber über Zeit
# Output: enso_coupling_analysis (auch für Macro Layer)
# ============================================================

# ── 9.1 ENSO-Klassen aus History extrahieren ─────────────────
enso_series = []
for s in snaps:
    ec = s.get('enso_context') or {}
    phase = ec.get('phase_class') or \
            s.get('layers', {}).get('L2_surface_zone', {}).get(
                'key_metrics', {}).get('ENSO_phase') or 'unknown'
    enso_series.append({
        'date':        s['_date'],
        'slot':        s['_slot'],
        'timestamp':   s['_mesz'],
        'phase_class': phase,
        'macro_phase': ec.get('macro_phase', 'unknown'),
        'direction':   ec.get('direction', 'unknown'),
        'event_risk':  ec.get('event_risk', 'unknown'),
        'nino34':      ec.get('nino34_anomaly_degC'),
        'oni':         ec.get('oni_3month_degC'),
        # Layer-Scores
        'L2': lscore(s, 'L2_surface_zone'),
        'L3': lscore(s, 'L3_atmosphere'),
        'L4': lscore(s, 'L4_ionosphere'),
        'L5': lscore(s, 'L5_global_electric_circuit'),
        'L6': lscore(s, 'L6_resonance_field'),
        'system_state': s['system_state'],
    })

phase_counts = Counter(e['phase_class'] for e in enso_series)
dominant_enso = phase_counts.most_common(1)[0][0] if phase_counts else 'unknown'

print('ENSO COUPLING TRACKER')
print('=' * 78)
print(f'  Snapshots mit ENSO-Daten: {sum(1 for e in enso_series if e["phase_class"] != "unknown")}/{len(enso_series)}')
print(f'  Dominante ENSO-Klasse:    {dominant_enso}')
print()
print('  ENSO-Klassen-Häufigkeit:')
for phase, c in phase_counts.most_common():
    pct = round(c / len(enso_series) * 100, 1)
    bar = '█' * c
    print(f'  {phase:<22} {bar} {c}× ({pct}%)')

# ── 9.2 Layer-Scores pro ENSO-Klasse ─────────────────────────
enso_layer_stats = {}
for phase in phase_counts:
    group = [e for e in enso_series if e['phase_class'] == phase]
    stats = {}
    for lname in ['L2', 'L3', 'L4', 'L5', 'L6']:
        vals = [e[lname] for e in group if e[lname] is not None]
        if vals:
            stats[lname] = {
                'mean': round(float(np.mean(vals)), 4),
                'std':  round(float(np.std(vals)),  4),
                'n':    len(vals),
            }
    enso_layer_stats[phase] = stats

print('\n  ── 9.2 Layer-Scores pro ENSO-Klasse ──')
print(f'  {"Phase":<22} {"L2":>7} {"L3":>7} {"L4":>7} {"L5":>7} {"L6":>7}')
for phase, stats in enso_layer_stats.items():
    row = f'  {phase:<22}'
    for lname in ['L2', 'L3', 'L4', 'L5', 'L6']:
        m = stats.get(lname, {}).get('mean')
        row += f' {m:>7.3f}' if m is not None else f' {"–":>7}'
    print(row)

# ── 9.3 L2 → L3 Verzögerung pro ENSO-Klasse ─────────────────
# Zieht L3 nach hohem L2 nach? (t+1, t+2 Snapshots)
enso_lag_analysis = {}
for phase in phase_counts:
    group_idx = [i for i, e in enumerate(enso_series) if e['phase_class'] == phase]
    l3_followups = []
    for i in group_idx:
        if i + 1 < len(enso_series):
            l2_now  = enso_series[i]['L2']
            l3_next = enso_series[i+1]['L3']
            if l2_now is not None and l3_next is not None:
                l3_followups.append((l2_now, l3_next))
    if len(l3_followups) >= 3:
        l2_arr = np.array([x[0] for x in l3_followups])
        l3_arr = np.array([x[1] for x in l3_followups])
        r = float(np.corrcoef(l2_arr, l3_arr)[0,1]) if l2_arr.std() > 0 else 0.0
        enso_lag_analysis[phase] = {
            'n':              len(l3_followups),
            'pearson_L2_L3_lag1': round(r, 4),
            'mean_L3_after':  round(float(l3_arr.mean()), 4),
        }

if enso_lag_analysis:
    print('\n  ── 9.3 L2 → L3 Lag+1 Korrelation pro ENSO-Klasse ──')
    for phase, info in enso_lag_analysis.items():
        strength = 'stark' if abs(info['pearson_L2_L3_lag1']) > 0.5 else \
                   'mittel' if abs(info['pearson_L2_L3_lag1']) > 0.3 else 'schwach'
        print(f'  {phase:<22}  r={info["pearson_L2_L3_lag1"]:+.3f}  '
              f'L3_mean_after={info["mean_L3_after"]:.3f}  '
              f'n={info["n"]}  ({strength})')

# ── 9.4 L5 nach L3-Aktivierung pro ENSO-Klasse ───────────────
enso_l5_followup = {}
for phase in phase_counts:
    l3_high_idx = [i for i, e in enumerate(enso_series)
                   if e['phase_class'] == phase and (e['L3'] or 0) >= 0.4]
    l5_after = []
    for i in l3_high_idx:
        if i + 1 < len(enso_series):
            v = enso_series[i+1]['L5']
            if v is not None:
                l5_after.append(v)
    if l5_after:
        enso_l5_followup[phase] = {
            'n_l3_high':      len(l3_high_idx),
            'n_l5_measured':  len(l5_after),
            'mean_L5_after':  round(float(np.mean(l5_after)), 4),
            'pct_l5_high':    round(sum(1 for v in l5_after if v >= 0.4) / len(l5_after) * 100, 1),
        }

if enso_l5_followup:
    print('\n  ── 9.4 L5 nach L3-Aktivierung (>= 0.4) ──')
    for phase, info in enso_l5_followup.items():
        print(f'  {phase:<22}  L5_mean={info["mean_L5_after"]:.3f}  '
              f'L5_high={info["pct_l5_high"]}%  '
              f'n={info["n_l5_measured"]}')

# ── 9.5 System-States pro ENSO-Klasse ────────────────────────
enso_state_dist = {}
for phase in phase_counts:
    group_states = [e['system_state'] for e in enso_series
                    if e['phase_class'] == phase]
    enso_state_dist[phase] = dict(Counter(group_states).most_common(3))

print('\n  ── 9.5 Häufigste System-States pro ENSO-Klasse ──')
for phase, states in enso_state_dist.items():
    top = ', '.join(f'{s.replace("_state","")}: {c}×' for s, c in states.items())
    print(f'  {phase:<22}  {top}')

# ── 9.6 ENSO-Trend ───────────────────────────────────────────
nino34_series = [(e['timestamp'], e['nino34']) for e in enso_series
                 if e['nino34'] is not None]
enso_trend = None
if len(nino34_series) >= 4:
    vals = np.array([v for _, v in nino34_series])
    slope = float(np.polyfit(np.arange(len(vals)), vals, 1)[0])
    enso_trend = round(slope, 5)
    direction  = 'warming' if slope > 0.001 else 'cooling' if slope < -0.001 else 'stable'
    print(f'\n  ── 9.6 Niño3.4 Trend ──')
    print(f'  Trend: {slope:+.5f} / Snapshot  →  {direction}')
    if nino34_series:
        print(f'  Aktuell: {nino34_series[-1][1]:+.2f}°C  '
              f'(Mittel: {float(vals.mean()):+.2f}°C)')

# ── 9.7 Haupt-Muster + Interpretation ───────────────────────
# Kernaussage für Makro-Layer
l2_warm   = enso_layer_stats.get(dominant_enso, {}).get('L2', {}).get('mean', 0) or 0
l3_warm   = enso_layer_stats.get(dominant_enso, {}).get('L3', {}).get('mean', 0) or 0
l5_warm   = enso_layer_stats.get(dominant_enso, {}).get('L5', {}).get('mean', 0) or 0

if l2_warm > 0.45 and l3_warm < 0.30:
    main_pattern     = 'L2 elevated while L3 remains weak'
    enso_hypothesis  = (f'ENSO {dominant_enso} acts as surface/ocean preparation, '
                        f'but downstream atmospheric activation is not yet persistent.')
elif l2_warm > 0.45 and l3_warm > 0.40:
    main_pattern     = 'L2 and L3 both elevated — surface-atmosphere coupling active'
    enso_hypothesis  = (f'ENSO {dominant_enso} drives both surface preparation and '
                        f'atmospheric activation. L5/L6 downstream expected.')
elif l2_warm < 0.30 and l3_warm < 0.30:
    main_pattern     = 'Low surface and atmosphere — suppressed regime'
    enso_hypothesis  = (f'ENSO {dominant_enso} associated with suppressed '
                        f'convection and low downstream activity.')
else:
    main_pattern     = 'Mixed signals — no clear surface-atmosphere pattern'
    enso_hypothesis  = f'ENSO {dominant_enso}: insufficient data for clear pattern.'

# Tests für Hypothesis Tracker
enso_tests = [
    f'Compare L3 score in repeated {dominant_enso} snapshots vs other phases.',
    'Check if L5 rises systematically after L3 activation (lag +1).',
    f'Check if L6 modulation increases during {dominant_enso} → el_nino transition.',
    'Track Niño3.4 trend: does warming correlate with higher downstream_score?',
]

# ── 9.8 Macro Handoff ────────────────────────────────────────
# Formatiert für spätere Integration in Macro Earth-System Layer
enso_coupling_analysis = {
    # Basis
    'dominant_enso_class':   dominant_enso,
    'phase_distribution':    dict(phase_counts),
    'nino34_trend_per_snap': enso_trend,

    # Layer-Scores pro Phase
    'layer_scores_by_phase': enso_layer_stats,

    # Lag-Analysen
    'l2_to_l3_lag1_by_phase':    enso_lag_analysis,
    'l5_after_l3_active_by_phase': enso_l5_followup,

    # State-Verteilung
    'state_distribution_by_phase': enso_state_dist,

    # Kernaussage
    'main_pattern':   main_pattern,
    'hypothesis':     enso_hypothesis,
    'tests':          enso_tests,

    # Makro-Layer Handoff
    'macro_handoff': {
        'system_axis':        'water_energy_stress',
        'enso_macro_phase':   enso_series[-1]['macro_phase'] if enso_series else 'unknown',
        'enso_event_risk':    enso_series[-1]['event_risk']  if enso_series else 'unknown',
        'surface_activation': round(l2_warm, 4),
        'atmospheric_activation': round(l3_warm, 4),
        'electric_activation':    round(l5_warm, 4),
        'coupling_pattern':   main_pattern,
        'feedback_risk':      ('medium_high' if l2_warm > 0.45 and l3_warm > 0.40
                               else 'low_medium'),
        'confidence':         evidence_level(len(enso_series)),
        'ready_for_macro_layer': len(enso_series) >= 10,
    },
}

print('\n  ── 9.7 Hauptmuster + Makro-Handoff ──')
print(f'  Muster:    {main_pattern}')
print(f'  Hypothese: {enso_hypothesis}')
print(f'\n  Makro-Layer Handoff:')
print(f'    system_axis:          {enso_coupling_analysis["macro_handoff"]["system_axis"]}')
print(f'    enso_macro_phase:     {enso_coupling_analysis["macro_handoff"]["enso_macro_phase"]}')
print(f'    surface_activation:   {enso_coupling_analysis["macro_handoff"]["surface_activation"]:.3f}')
print(f'    atmospheric_activ.:   {enso_coupling_analysis["macro_handoff"]["atmospheric_activation"]:.3f}')
print(f'    electric_activation:  {enso_coupling_analysis["macro_handoff"]["electric_activation"]:.3f}')
print(f'    feedback_risk:        {enso_coupling_analysis["macro_handoff"]["feedback_risk"]}')
print(f'    ready_for_macro:      {enso_coupling_analysis["macro_handoff"]["ready_for_macro_layer"]}')

## 9. State-Transition Matrix

Welcher Zustand folgt auf welchen? Wir betrachten benachbarte Snapshots in chronologischer Reihenfolge.

In [ ]:
transitions = []
for i in range(len(snaps) - 1):
    from_state = snaps[i]['system_state']
    to_state   = snaps[i+1]['system_state']
    if from_state != to_state:
        # nur echte Transitionen (kein 'stay')
        transitions.append({
            'from': from_state,
            'to':   to_state,
            'time': snaps[i+1]['_mesz'].strftime('%m-%d %H:%M'),
        })

# Matrix
trans_matrix = defaultdict(lambda: defaultdict(int))
for t in transitions:
    trans_matrix[t['from']][t['to']] += 1

print('STATE TRANSITIONS')
print('=' * 78)
print(f'  Echte Übergänge: {len(transitions)}')
print()
for t in transitions:
    fs = t['from'].replace('_state', '')
    ts = t['to'].replace('_state', '')
    print(f'  {t["time"]:>12}  {fs:<28} → {ts}')

print()
print('TRANSITION COUNTS')
print('=' * 78)
for from_s, to_dict in trans_matrix.items():
    fs = from_s.replace('_state', '')
    for to_s, count in to_dict.items():
        ts = to_s.replace('_state', '')
        print(f'  {fs:<28} → {ts:<28} {count}×')

transition_summary = {
    'total_transitions': len(transitions),
    'transitions':       transitions,
    'matrix':            {k: dict(v) for k, v in trans_matrix.items()},
}

In [ ]:
# ============================================================
# EVIDENZ-BEWERTUNG — wissenschaftlich vorsichtig
# ============================================================

def evidence_level(n):
    """Klassifiziert Stichprobengröße in Evidenz-Stufe"""
    if n < 10:  return 'exploratory_signal'
    if n < 30:  return 'testable'
    if n < 100: return 'moderate_evidence'
    return 'robust_candidate'

def make_hypothesis(id, question, status, evidence_str, n, next_step):
    """Erstellt Hypothese mit automatischer Evidenz-Bewertung"""
    ev_level = evidence_level(n)
    # Status nach unten korrigieren wenn n zu klein
    corrected_status = status
    if ev_level == 'exploratory_signal' and status in ('confirmed', 'likely'):
        corrected_status = 'open'
    elif ev_level == 'testable' and status == 'confirmed':
        corrected_status = 'likely'
    return {
        'id':               id,
        'question':         question,
        'status':           corrected_status,
        'status_original':  status,   # vor Korrektur
        'evidence':         evidence_str,
        'n':                n,
        'evidence_level':   ev_level,
        'next_step':        next_step,
    }

## 10. Hypothesis Tracker

Hypothesen werden **regelbasiert** generiert: Bedingung erfüllt → Hypothese aktiv mit Evidenzstand.

In [ ]:
hypotheses = []

# H1: Carnegie Cycle ─────────────────────────────────────────
n_evenings_total = sum(1 for s in snaps if s['_slot'] == 'evening')
n_anomal_evening = sum(1 for s in snaps
                       if s['_slot'] == 'evening' and s['system_state'] == 'anomalous_resonance_state')
n_anomal_morning = sum(1 for s in snaps
                       if s['_slot'] == 'morning' and s['system_state'] == 'anomalous_resonance_state')
n_anomal_total = sum(1 for s in snaps if s['system_state'] == 'anomalous_resonance_state')

if n_anomal_total > 0:
    pct_in_evening = round(n_anomal_evening / n_anomal_total * 100, 1)
    if pct_in_evening >= 90:
        status = 'confirmed'
    elif pct_in_evening >= 70:
        status = 'likely'
    else:
        status = 'mixed'
    hypotheses.append({
        'id': 'H1',
        'question': 'Tritt anomalous_resonance_state ausschließlich abends (>= 18 MESZ) auf?',
        'status':   status,
        'evidence': f'{n_anomal_evening}/{n_anomal_total} Anomalous-Events im Abend-Slot ({pct_in_evening}%). Morning: {n_anomal_morning}.',
        'next_step': 'Mehr Snapshots; ggf. mittäglichen Slot ergänzen für Bestätigung.',
    })

# H2: ΔL3 Schwellenwert ──────────────────────────────────────
if anomal_dl3 and seasonal_dl3:
    overlap = max(seasonal_dl3) >= min(anomal_dl3)
    status  = 'likely' if not overlap else 'mixed'
# H1: Carnegie Cycle
if n_anomal_total > 0:
    pct_in_evening = round(n_anomal_evening / n_anomal_total * 100, 1)
    status = 'confirmed' if pct_in_evening >= 90 else 'likely' if pct_in_evening >= 70 else 'mixed'
    hypotheses.append(make_hypothesis(
        id       = 'H1',
        question = 'Tritt anomalous_resonance_state ausschließlich abends (>= 18 MESZ) auf?',
        status   = status,
        evidence_str = f'{n_anomal_evening}/{n_anomal_total} Anomalous-Events im Abend-Slot ({pct_in_evening}%). Morning: {n_anomal_morning}.',
        n        = n_anomal_total,
        next_step = 'Mehr Snapshots; ggf. mittäglichen Slot ergänzen.',
    ))

# H3: L2-L3 Paradox ──────────────────────────────────────────
if pearson_l2_l3 < -0.2:
    hypotheses.append({
        'id': 'H3',
        'question': 'Existiert ein L2→L3 Aktivierungsparadox (steigendes L2, fallendes L3)?',
        'status':   'open',
        'evidence': f'Pearson L2 vs L3 = {pearson_l2_l3:+.3f}. L2 Trend {l2_trend:+.5f} (steigend). L3 Trend {l3_trend:+.5f}. Gap wächst um {gap_trend:+.5f}/Snapshot.',
        'next_step': 'Verzögerte Korrelation L2(t) vs L3(t+24h, t+48h, t+7d) prüfen. Möglich: Sättigungseffekt oder fehlender Trigger (Windscherung, Kaltlufteinbruch) im L3-Modell.',
    })

# H4: Carnegie Amplituden-Modulator
if 'correlations' in carnegie_amplitude:
    top_var, top_r = max(carnegie_amplitude['correlations'].items(), key=lambda x: abs(x[1]))
    if abs(top_r) > 0.5:
        hypotheses.append(make_hypothesis(
            id       = 'H4',
            question = f'Moduliert {top_var} die Carnegie-Amplitude (L5 abends)?',
            status   = 'likely' if abs(top_r) > 0.7 else 'open',
            evidence_str = f'Pearson L5_evening vs {top_var} = {top_r:+.3f} über {carnegie_amplitude["n_evenings"]} Abende.',
            n        = carnegie_amplitude['n_evenings'],
            next_step = 'Korrelation in größerer Stichprobe bestätigen.',
        ))

# H_combined: Combined Score vs ΔL3 allein
if combined_threshold is not None and anomal_combined:
    hypotheses.append(make_hypothesis(
        id       = 'H_combined',
        question = 'Ist combined_activation_score (ΔL3+L5+L6) besser als ΔL3 allein?',
        status   = 'likely' if not overlap_combined and overlap_dl3 else 'open',
        evidence_str = (f'combined Overlap={overlap_combined} vs ΔL3 Overlap={overlap_dl3}. '
                        f'combined Schwelle={combined_threshold:.4f}.'),
        n        = len(complete_pairs),
        next_step = 'Mehr Tagespaare für robuste Trennung. ROC-Analyse ab n>=20.',
    ))

# H5: Operator-basierte Hypothesen ───────────────────────────
if operator_analysis.get('status') == 'active':
    if operator_analysis.get('operator_dl3_correlation'):
        for op, r in operator_analysis['operator_dl3_correlation'].items():
            if abs(r) > 0.5:
                short = op.replace('_operator', '')
                hypotheses.append({
                    'id': f'H_op_{short}',
                    'question': f'Ist der {short} Operator ein Prädiktor für ΔL3-Aktivierung?',
                    'status':   'likely' if abs(r) > 0.7 else 'open',
                    'evidence': f'Pearson {op} vs ΔL3 = {r:+.3f}.',
                    'next_step': 'Bestätigung mit mehr Operator-Snapshots.',
                })

# H6: Persistente Background-Tags ───────────────────────────
all_tags = []
for s in snaps:
    all_tags.extend(s.get('event_tags', []))
tag_freq = Counter(all_tags)
persistent_tags = [t for t, c in tag_freq.items()
                   if c == len(snaps) and not t.startswith('state_')]
if persistent_tags:
    hypotheses.append({
        'id': 'H6',
        'question': 'Welche Tags sind persistenter Hintergrund vs welche sind Signal?',
        'status':   'open',
        'evidence': f'In 100% aller Snapshots: {", ".join(persistent_tags)}. Diese sind kein Aktivierungssignal sondern Saison-Hintergrund.',
        'next_step': 'Hintergrund-Tags von Layer-7-Tag-Liste trennen oder als baseline_tags markieren.',
    })

# H7: Lead-Lag-Operatoren
if operator_analysis.get('lead_lag'):
    for key, info in operator_analysis['lead_lag'].items():
        if abs(info['pearson']) > 0.6:
            hypotheses.append({
                'id': f'H_leadlag_{key}',
                'question': f'Sagt {key.split("__")[0]} den Wert {key.split("__")[1]} voraus?',
                'status':   'likely' if abs(info['pearson']) > 0.7 else 'open',
                'evidence': f'Pearson = {info["pearson"]:+.3f} (n={info["n_pairs"]}, Lag {info["lag_snapshots"]} Snapshots).',
                'next_step': 'Bestätigung mit mehr Snapshots; Schwelle für Vorhersage definieren.',
            })

# H8: Dominantes Operator-Regime
if operator_analysis.get('operator_regimes'):
    rc = operator_analysis['operator_regimes']['counts']
    if rc:
        top_regime, top_count = max(rc.items(), key=lambda x: x[1])
        pct = round(top_count / ops_count * 100, 1)
        if pct >= 40:
            hypotheses.append({
                'id': 'H8',
                'question': f'Ist "{top_regime}" das dominante Operator-Regime?',
                'status':   'likely' if pct >= 60 else 'open',
                'evidence': f'{top_count}/{ops_count} Snapshots ({pct}%) im Regime "{top_regime}".',
                'next_step': 'Übergangsmuster zwischen Regimen analysieren (Layer 9?).',
            })

# Ausgabe
icon = {'confirmed':'✅','likely':'🔶','open':'❓','mixed':'⚠️'}
ev_icon = {'exploratory_signal':'🔬','testable':'🧪','moderate_evidence':'📊','robust_candidate':'✔️'}
print('HYPOTHESIS TRACKER')
print('=' * 78)
for h in hypotheses:
    ic = icon.get(h['status'], '?')
    ev = ev_icon.get(h.get('evidence_level',''), '')
    corrected = ' (korrigiert)' if h.get('status') != h.get('status_original') else ''
    print(f'\n  {ic} {h["id"]}: {h["question"]}')
    print(f'     Status:   {h["status"]}{corrected}  {ev} {h.get("evidence_level","")}  (n={h.get("n","?")})')
    print(f'     Evidenz:  {h["evidence"]}')
    print(f'     → {h["next_step"]}')

## 11. Visualisierungen

In [ ]:
# Plot 1: ΔL3 pro Tag mit State-Markierung
if dl3_data:
    dates_   = [r['date'] for r in dl3_data]
    dl3_vals = [r['delta_L3'] for r in dl3_data]
    colors   = ['#e74c3c' if r['evening_state'] == 'anomalous_resonance_state' else '#3498db'
                for r in dl3_data]

    fig1 = go.Figure()
    fig1.add_trace(go.Bar(
        x=dates_, y=dl3_vals, marker_color=colors,
        text=[f'{v:+.3f}' for v in dl3_vals], textposition='outside',
        textfont=dict(color='white'),
    ))
    if dl3_threshold is not None:
        fig1.add_hline(y=dl3_threshold, line_dash='dash', line_color='#f39c12',
                       annotation_text=f'Schwelle ΔL3 = {dl3_threshold:+.3f}',
                       annotation_position='top right',
                       annotation_font_color='#f39c12')
    fig1.update_layout(
        title=dict(text='ΔL3 (L3 abends − L3 morgens) pro Tag',
                   font=dict(size=14, color='white')),
        xaxis=dict(title='Datum', color='white', gridcolor='#222244'),
        yaxis=dict(title='ΔL3', color='white', gridcolor='#222244'),
        height=350, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
        showlegend=False, margin=dict(l=60, r=40, t=55, b=50),
    )
    fig1.show()

In [ ]:
# ============================================================
# Plot 2: Field Operators — Multi-Panel
# Panel A: Aktuelle Werte + Trend-Indikator
# Panel B: Operator-Zeitreihen (Verlauf)
# Panel C: Operator-Regime Häufigkeit
# ============================================================

if operator_analysis.get('status') == 'active' and operator_analysis.get('latest_ranking'):
    ranking = operator_analysis['latest_ranking']
    trends_data = operator_analysis.get('operator_trends', {})

    fig2 = make_subplots(
        rows=2, cols=2,
        specs=[[{'colspan': 2}, None],
               [{}, {}]],
        subplot_titles=(
            'Aktuelle Operatoren mit Trend',
            'Operator-Zeitreihen',
            'Operator-Regime Häufigkeit',
        ),
        row_heights=[0.42, 0.58],
        vertical_spacing=0.18,
        horizontal_spacing=0.12,
    )

    # ── Panel A: Aktuelle Werte mit Trend-Pfeilen ────────────
    names_short = [r['operator'].replace('_operator', '').replace('_', ' ') for r in ranking]
    scores      = [r['score'] for r in ranking]
    colors      = ['#2ecc71' if s < 0.3 else '#f39c12' if s < 0.6 else '#e74c3c' for s in scores]

    # Trend-Pfeile + Delta-Text aus operator_trends
    arrow_map = {'rising': '↑', 'falling': '↓', 'stable': '→', 'unknown': '?'}
    text_labels = []
    for r in ranking:
        t = trends_data.get(r['operator'])
        if t and t.get('delta') is not None:
            arrow = arrow_map.get(t.get('direction', 'unknown'), '?')
            text_labels.append(f'{r["score"]:.3f}  {arrow} Δ{t["delta"]:+.3f}')
        else:
            text_labels.append(f'{r["score"]:.3f}')

    fig2.add_trace(go.Bar(
        x=scores, y=names_short, orientation='h',
        marker_color=colors, opacity=0.85,
        text=text_labels, textposition='outside',
        textfont=dict(color='white', size=11),
        showlegend=False,
    ), row=1, col=1)
    fig2.add_vline(x=0.3, line_dash='dot', line_color='#888780', row=1, col=1)
    fig2.add_vline(x=0.6, line_dash='dot', line_color='#f39c12', row=1, col=1)
    fig2.update_xaxes(range=[0, 1.25], gridcolor='#222244', row=1, col=1)

    # ── Panel B: Operator-Zeitreihen ─────────────────────────
    op_palette = {
        'thermal_operator':                '#e74c3c',
        'electric_operator':               '#f39c12',
        'ionization_operator':             '#9b59b6',
        'geomagnetic_operator':            '#3498db',
        'resonance_model_operator':        '#1abc9c',
        'tidal_gravity_operator':          '#95a5a6',
        'cross_layer_activation_operator': '#e67e22',
    }

    ops_timestamps = [s['_mesz'] for s in ops_snaps]

    for op in OP_NAMES:
        vals = [_get_op(s, op) for s in ops_snaps]
        if all(v is None for v in vals):
            continue
        y_plot = [v if v is not None else None for v in vals]
        fig2.add_trace(go.Scatter(
            x=ops_timestamps, y=y_plot,
            name=op.replace('_operator', ''),
            mode='lines+markers',
            line=dict(color=op_palette.get(op, '#ffffff'), width=2),
            marker=dict(size=6),
            connectgaps=False,
        ), row=2, col=1)
    fig2.add_hline(y=HIGH, line_dash='dot', line_color='#e74c3c',
                   line_width=1, row=2, col=1)
    fig2.add_hline(y=LOW,  line_dash='dot', line_color='#888780',
                   line_width=1, row=2, col=1)
    fig2.update_xaxes(gridcolor='#222244', row=2, col=1)
    fig2.update_yaxes(title_text='Score', range=[0, 1.05],
                      gridcolor='#222244', row=2, col=1)

    # ── Panel C: Operator-Regime Häufigkeit ──────────────────
    regime_counts = operator_analysis.get('operator_regimes', {}).get('counts', {})
    if regime_counts:
        regime_palette = {
            'surface_prepared_delayed': '#e67e22',
            'electric_coupling_mode':   '#f39c12',
            'space_weather_mode':       '#9b59b6',
            'quiet_background':         '#2ecc71',
            'mixed':                    '#888780',
        }
        sorted_regimes = sorted(regime_counts.items(), key=lambda x: -x[1])
        reg_names  = [r[0] for r in sorted_regimes]
        reg_counts = [r[1] for r in sorted_regimes]
        reg_colors = [regime_palette.get(r, '#666666') for r in reg_names]

        fig2.add_trace(go.Bar(
            x=reg_counts, y=reg_names, orientation='h',
            marker_color=reg_colors, opacity=0.85,
            text=[f'{c}× ({round(c/ops_count*100)}%)' for c in reg_counts],
            textposition='outside', textfont=dict(color='white', size=10),
            showlegend=False,
        ), row=2, col=2)
        fig2.update_xaxes(gridcolor='#222244', row=2, col=2,
                          range=[0, max(reg_counts) * 1.4])
    else:
        fig2.add_annotation(
            text='keine Regime-Daten', x=0.5, y=0.5,
            xref='x4', yref='y4', showarrow=False,
            font=dict(color='#888', size=11),
            row=2, col=2,
        )

    # ── Layout ───────────────────────────────────────────────
    fig2.update_layout(
        height=700,
        plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
        font=dict(color='white'),
        margin=dict(l=180, r=60, t=70, b=40),
        legend=dict(
            orientation='h', y=-0.08, x=0.5, xanchor='center',
            bgcolor='rgba(0,0,0,0)', font=dict(color='white', size=10),
        ),
        title=dict(
            text=f'Field Operators — Coverage {coverage_pct}% ({coverage_regime})',
            font=dict(size=14, color='white'),
            x=0.5, xanchor='center',
        ),
    )
    for r, c in [(1,1),(2,1),(2,2)]:
        fig2.update_yaxes(color='white', tickfont=dict(size=10), row=r, col=c)
        fig2.update_xaxes(color='white', tickfont=dict(size=10), row=r, col=c)

    fig2.show()

In [ ]:
# Plot 3: System-State Timeline + L5-Verlauf
ts_x   = [s['_mesz'] for s in snaps]
l5_y   = [lscore(s, 'L5_global_electric_circuit') for s in snaps]
l3_y   = [lscore(s, 'L3_atmosphere') for s in snaps]
states = [s['system_state'] for s in snaps]

state_colors = {
    'seasonal_transition_state':   '#3498db',
    'anomalous_resonance_state':   '#e74c3c',
    'cavity_condition_shift_state':'#f39c12',
    'normal_background_state':     '#2ecc71',
}

fig3 = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                     subplot_titles=('L5 (GEC) und L3 (Atmosphäre)', 'System-State'))
fig3.add_trace(go.Scatter(x=ts_x, y=l5_y, name='L5', mode='lines+markers',
                          line=dict(color='#e74c3c'), marker=dict(size=8)), row=1, col=1)
fig3.add_trace(go.Scatter(x=ts_x, y=l3_y, name='L3', mode='lines+markers',
                          line=dict(color='#3498db'), marker=dict(size=8)), row=1, col=1)
state_y = [list(state_colors.keys()).index(st) if st in state_colors else 0 for st in states]
fig3.add_trace(go.Scatter(
    x=ts_x, y=state_y, mode='markers', showlegend=False,
    marker=dict(size=12, color=[state_colors.get(st, '#888') for st in states]),
    text=states, hovertemplate='%{text}<extra></extra>',
), row=2, col=1)
fig3.update_layout(
    height=520, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white'),
    legend=dict(bgcolor='rgba(0,0,0,0)', font=dict(color='white')),
    margin=dict(l=60, r=40, t=60, b=40),
)
fig3.update_xaxes(gridcolor='#222244', color='white')
fig3.update_yaxes(gridcolor='#222244', color='white')
fig3.update_yaxes(tickvals=list(range(len(state_colors))),
                  ticktext=[s.replace('_state','') for s in state_colors.keys()],
                  row=2, col=1)
fig3.show()

In [ ]:
# ============================================================
# 12. CAVITY EVENT STUDY
# Analysiert jeden cavity_condition_shift_state im Kontext
# ============================================================

cavity_events = []
for i, s in enumerate(snaps):
    if s['system_state'] != 'cavity_condition_shift_state':
        continue

    t_minus = snaps[i-1] if i > 0 else None
    t_zero  = s
    t_plus  = snaps[i+1] if i < len(snaps)-1 else None

    def snap_layer(snap, lname):
        if snap is None: return None
        return snap['layers'].get(lname, {}).get('score')

    def snap_op(snap, op):
        if snap is None: return None
        ops = snap.get('field_operators') or {}
        o = ops.get(op) or ops.get('resonance_operator')
        if isinstance(o, dict): return o.get('score')
        return None

    def snap_coupling(snap, from_l, to_l):
        if snap is None: return None
        for c in snap.get('couplings', []):
            if c['from'] == from_l and c['to'] == to_l:
                return c['strength']
        return None

    # Layer-Verläufe
    layers_track = {}
    for lname in ['L3_atmosphere', 'L4_ionosphere',
                  'L5_global_electric_circuit', 'L6_resonance_field']:
        layers_track[lname] = {
            't-1': snap_layer(t_minus, lname),
            't':   snap_layer(t_zero,  lname),
            't+1': snap_layer(t_plus,  lname),
        }

    # Kopplungen bei t
    l4_l6 = snap_coupling(t_zero, 'L4_ionosphere', 'L6_resonance_field')
    l5_l6 = snap_coupling(t_zero, 'L5_global_electric_circuit', 'L6_resonance_field')

    # Operators bei t
    res_op   = snap_op(t_zero, 'resonance_model_operator')
    cross_op = snap_op(t_zero, 'cross_layer_activation_operator')

    # Downstream-Bestätigung: L5+L6 nach Cavity-Event
    l5_after = snap_layer(t_plus, 'L5_global_electric_circuit')
    l6_after = snap_layer(t_plus, 'L6_resonance_field')

    # Klassifikation
    if t_plus is not None:
        next_state = t_plus['system_state']
        if next_state == 'anomalous_resonance_state':
            outcome = 'cavity_shift_precursor'
        elif next_state == 'cavity_condition_shift_state':
            outcome = 'cavity_shift_persistent'
        elif (l5_after or 0) < 0.3 and (l6_after or 0) < 0.3:
            outcome = 'cavity_shift_failed'
        else:
            outcome = 'cavity_shift_neutral'
    else:
        outcome = 'unknown_no_followup'

    event = {
        'timestamp':     t_zero['_mesz'].strftime('%Y-%m-%d %H:%M'),
        'slot':          t_zero['_slot'],
        'outcome':       outcome,
        'state_t_minus': t_minus['system_state'].replace('_state','') if t_minus else '—',
        'state_t':       'cavity_condition_shift',
        'state_t_plus':  t_plus['system_state'].replace('_state','') if t_plus else '—',
        'layers':        layers_track,
        'couplings': {
            'L4_to_L6': round(l4_l6, 3) if l4_l6 is not None else None,
            'L5_to_L6': round(l5_l6, 3) if l5_l6 is not None else None,
        },
        'operators': {
            'resonance_model':        round(res_op,   3) if res_op   is not None else None,
            'cross_layer_activation': round(cross_op, 3) if cross_op is not None else None,
        },
    }
    cavity_events.append(event)

outcome_counts = Counter(e['outcome'] for e in cavity_events)

print('CAVITY EVENT STUDY')
print('=' * 78)
print(f'  Cavity-Events gesamt: {len(cavity_events)}')
print()
for e in cavity_events:
    print(f'  ── {e["timestamp"]} ({e["slot"]}) ──')
    print(f'     {e["state_t_minus"]} → cavity_shift → {e["state_t_plus"]}')
    print(f'     Outcome: {e["outcome"]}')
    for lname, vals in e['layers'].items():
        short = lname.split('_')[0]
        tm  = f'{vals["t-1"]:.3f}' if vals['t-1'] is not None else '  —  '
        t0  = f'{vals["t"]:.3f}'   if vals['t']   is not None else '  —  '
        tp  = f'{vals["t+1"]:.3f}' if vals['t+1'] is not None else '  —  '
        print(f'     {short:<4}  t-1={tm}  t={t0}  t+1={tp}')
    print(f'     L4→L6={e["couplings"]["L4_to_L6"]}  '
          f'L5→L6={e["couplings"]["L5_to_L6"]}  '
          f'res_op={e["operators"]["resonance_model"]}  '
          f'cross_op={e["operators"]["cross_layer_activation"]}')
    print()

print('Outcome-Verteilung:')
for outcome, c in outcome_counts.most_common():
    print(f'  {outcome:<30} {c}×')

In [ ]:
# ============================================================
# 13. HYPOTHESIS FACTORY (v2)
# Wissenschaftlich vorsichtig: Status ≠ Evidenz
# ============================================================

import os

CANDIDATES_DIR = STATES_REGISTRY / 'hypothesis_candidates'
REGISTRY_FILE  = STATES_REGISTRY / 'hypothesis_registry.json'
os.makedirs(CANDIDATES_DIR, exist_ok=True)

# ── Hilfsfunktionen ──────────────────────────────────────────

def evidence_level(n):
    if n is None:     return 'unknown'
    if n < 10:        return 'exploratory_signal'
    if n < 30:        return 'testable'
    if n < 100:       return 'moderate_evidence'
    return 'robust_candidate'

def promotion_blocked_reason(n, pattern_status, review_status):
    """Erklärt warum eine Hypothese noch nicht ins Modell darf"""
    if review_status != 'reviewed':
        return 'not_manually_reviewed'
    if n is None or n < 10:
        return 'insufficient_sample'
    if pattern_status == 'absent':
        return 'pattern_not_detected'
    return None

def make_candidate(
    id, type, question,
    pattern_status,       # detected / absent / unclear
    n, source_metric,
    evidence_str,
    next_step,
    effect_size=None,
    pearson=None,
    lag=None,
    mechanism=None,
    priority='medium',    # high / medium / low / quarantine
    category='core',      # core / diagnostic / exploratory_leadlag / waiting
):
    ev_level = evidence_level(n)

    # Fehlende Evidenz → sofort als candidate ohne Status
    if n is None or ev_level == 'unknown':
        pat_status = 'unclear'
        blocked    = 'missing_evidence_metadata'
    else:
        pat_status = pattern_status
        blocked    = promotion_blocked_reason(n, pat_status, 'unreviewed')

    return {
        'id':                       id,
        'type':                     type,
        'question':                 question,
        'pattern_status':           pat_status,
        'evidence_level':           ev_level,
        'n':                        n,
        'effect_size':              effect_size,
        'pearson':                  pearson,
        'lag':                      lag,
        'source_metric':            source_metric,
        'mechanism':                mechanism,
        'evidence':                 evidence_str,
        'next_step':                next_step,
        'priority':                 priority,
        'category':                 category,
        'review_status':            'unreviewed',
        'include_in_report':        False,
        'include_in_model_logic':   False,
        'promotion_blocked_reason': blocked,
        'generated_at':             RUN_TS,
        'source_snapshots':         len(snaps),
    }

# ── Kandidaten definieren ────────────────────────────────────

factory_candidates = []

# ── CORE CANDIDATES ──────────────────────────────────────────

# H_combined — höchste Priorität
if combined_threshold is not None and anomal_combined:
    factory_candidates.append(make_candidate(
        id             = 'H_combined',
        type           = 'predictive_candidate',
        question       = 'Sagt combined_activation_score (ΔL3+L5+L6) anomalous_resonance besser voraus als ΔL3 allein?',
        pattern_status = 'detected' if not overlap_combined else 'unclear',
        n              = len(complete_pairs),
        source_metric  = 'combined_activation_score',
        effect_size    = round(min(anomal_combined) - max(seasonal_combined), 4) if not overlap_combined else 0.0,
        evidence_str   = (f'combined Overlap={overlap_combined} (Schwelle={combined_threshold:.4f}). '
                          f'ΔL3 Overlap={overlap_dl3}. '
                          f'anomalous mean={np.mean(anomal_combined):.3f}, '
                          f'seasonal mean={np.mean(seasonal_combined):.3f}.'),
        mechanism      = 'ΔL3 misst atmosphärische Aktivierung, L5/L6 downstream-Bestätigung — kombiniert robusteres Signal',
        next_step      = 'ROC-Kurve ab n>=20 Tagespaaren. Schwelle formal testen.',
        priority       = 'high',
        category       = 'core',
    ))

# H1 — Carnegie Tageszeitmuster
if n_anomal_total > 0:
    pct = round(n_anomal_evening / n_anomal_total * 100, 1)
    factory_candidates.append(make_candidate(
        id             = 'H1',
        type           = 'temporal_pattern',
        question       = 'Tritt anomalous_resonance_state ausschließlich im Abend-Slot (>= 18 MESZ) auf?',
        pattern_status = 'detected' if pct >= 70 else 'unclear',
        n              = n_anomal_total,
        source_metric  = 'slot_annotation',
        evidence_str   = f'{n_anomal_evening}/{n_anomal_total} Anomalous-Events abends ({pct}%). Morning: {n_anomal_morning}.',
        mechanism      = 'Carnegie-Tagesgang: maximale GEC-Aktivität 18–20 UTC',
        next_step      = 'Mehr Snapshots. Mittags-Slot ergänzen zur Kontrolle.',
        priority       = 'high',
        category       = 'core',
    ))

# H3 — L2→L3 Paradox
if pearson_l2_l3 < -0.1:
    factory_candidates.append(make_candidate(
        id             = 'H3',
        type           = 'structural_paradox',
        question       = 'Leben L2 (Wochen-Trend) und L3 (Tagesrhythmus) auf verschiedenen Zeitskalen?',
        pattern_status = 'detected' if pearson_l2_l3 < -0.2 else 'unclear',
        n              = len(snaps),
        source_metric  = 'pearson_L2_L3',
        pearson        = round(pearson_l2_l3, 4),
        evidence_str   = (f'Pearson L2 vs L3 = {pearson_l2_l3:+.4f}. '
                          f'L2 Trend={l2_trend:+.5f}/Snap, L3 Trend={l3_trend:+.5f}/Snap. '
                          f'Gap-Trend={gap_trend:+.5f}/Snap.'),
        mechanism      = 'L2 = SST/ENSO (Wochen-Skala), L3 = konvektive Aktivität (Tages-Skala)',
        next_step      = 'Verzögerte Korrelation L2(t) vs L3(t+24h/48h/7d) prüfen.',
        priority       = 'high',
        category       = 'core',
    ))

# H4 — Carnegie Amplitudenmodulator
if 'correlations' in carnegie_amplitude:
    top_var, top_r = max(carnegie_amplitude['correlations'].items(), key=lambda x: abs(x[1]))
    if abs(top_r) > 0.3:
        factory_candidates.append(make_candidate(
            id             = 'H4',
            type           = 'amplitude_modulator',
            question       = f'Moduliert {top_var} die Carnegie-Amplitude (L5 abends)?',
            pattern_status = 'detected' if abs(top_r) > 0.5 else 'unclear',
            n              = carnegie_amplitude['n_evenings'],
            source_metric  = f'pearson_L5evening_vs_{top_var}',
            pearson        = round(top_r, 4),
            evidence_str   = (f'Pearson L5_evening vs {top_var} = {top_r:+.3f} '
                              f'über {carnegie_amplitude["n_evenings"]} Abende.'),
            mechanism      = 'Gewitteraktivität (L3) treibt GEC-Generator → L5-Amplitude',
            next_step      = 'Mehr Abend-Snapshots. Alle Kandidaten-Variablen tabellarisch vergleichen.',
            priority       = 'high',
            category       = 'core',
        ))

# H_op_cross_layer — Übergangsdynamik
if operator_analysis.get('operator_dl3_correlation'):
    cross_r = operator_analysis['operator_dl3_correlation'].get('cross_layer_activation_operator')
    if cross_r is not None:
        factory_candidates.append(make_candidate(
            id             = 'H_op_cross_layer',
            type           = 'operator_predictor',
            question       = 'Ist cross_layer_activation_operator ein Vorläufer für ΔL3-Aktivierung?',
            pattern_status = 'detected' if abs(cross_r) > 0.4 else 'unclear',
            n              = len(pairs_with_ops),
            source_metric  = 'cross_layer_activation_operator',
            pearson        = round(cross_r, 4),
            evidence_str   = f'Pearson cross_layer_activation vs ΔL3 = {cross_r:+.3f} (n={len(pairs_with_ops)}).',
            mechanism      = 'Gap zwischen vorbereiteter und nicht-aktivierter Schicht als Aktivierungsspannung',
            next_step      = 'Mehr Operator-Snapshots. Lead-Lag bei t+1 und t+2 prüfen.',
            priority       = 'high',
            category       = 'core',
        ))

# ── DIAGNOSTIC CANDIDATES ────────────────────────────────────

# H6 — Persistente Hintergrund-Tags
all_tags_flat = [t for s in snaps for t in s.get('event_tags', [])]
tag_freq_all  = Counter(all_tags_flat)
persistent    = [t for t, c in tag_freq_all.items()
                 if c == len(snaps) and not t.startswith('state_')]
if persistent:
    factory_candidates.append(make_candidate(
        id             = 'H6',
        type           = 'tag_hygiene',
        question       = 'Welche Tags sind Saison-Hintergrund und kein echtes Aktivierungssignal?',
        pattern_status = 'detected',
        n              = len(snaps),
        source_metric  = 'event_tag_frequency',
        evidence_str   = f'In 100% aller Snapshots: {", ".join(persistent)}.',
        mechanism      = 'Dauerpräsente Tags haben keinen diskriminativen Wert für State-Vorhersage',
        next_step      = 'baseline_tags in Layer 7 final festlegen. Signal-Tags separat auswerten.',
        priority       = 'medium',
        category       = 'diagnostic',
    ))

# H_cavity — Cavity Gate Outcome-Muster
if cavity_events:
    n_precursor = outcome_counts.get('cavity_shift_precursor', 0)
    n_failed    = outcome_counts.get('cavity_shift_failed', 0)
    n_total_cav = len(cavity_events)
    factory_candidates.append(make_candidate(
        id             = 'H_cavity',
        type           = 'precursor_pattern',
        question       = 'Folgt auf cavity_condition_shift_state häufiger eine Aktivierung als ein Fehlschlag?',
        pattern_status = 'detected' if n_precursor > n_failed else 'unclear',
        n              = n_total_cav,
        source_metric  = 'cavity_gate_outcome',
        evidence_str   = (f'{n_total_cav} Cavity-Events: '
                          f'precursor={n_precursor}, failed={n_failed}, '
                          f'neutral={outcome_counts.get("cavity_shift_neutral",0)}.'),
        mechanism      = 'Cavity-Verschiebung als elektromagnetischer Vorbote',
        next_step      = 'Mehr Cavity-Events sammeln. L4/L6-Kopplungsstärke als Schwelle testen.',
        priority       = 'medium',
        category       = 'diagnostic',
    ))

# ── EXPLORATORY LEAD-LAG POOL ────────────────────────────────

if operator_analysis.get('lead_lag'):
    for key, info in operator_analysis['lead_lag'].items():
        r   = info['pearson']
        n_l = info['n_pairs']
        if abs(r) < 0.4: continue   # nur interessante behalten
        factory_candidates.append(make_candidate(
            id             = f'H_leadlag_{key}',
            type           = 'lead_lag_signal',
            question       = f'Sagt {key.split("__")[0]}(t) den Wert {key.split("__")[1]} voraus?',
            pattern_status = 'detected' if abs(r) > 0.5 else 'unclear',
            n              = n_l,
            source_metric  = key,
            pearson        = round(r, 4),
            lag            = info['lag_snapshots'],
            evidence_str   = (f'Pearson={r:+.3f}, n={n_l}, '
                              f'Lag={info["lag_snapshots"]} Snapshots (~{info["lag_snapshots"]*6}h).'),
            mechanism      = None,   # muss manuell ergänzt werden
            next_step      = 'Mechanismus identifizieren. n >= 10 abwarten.',
            priority       = 'low',
            category       = 'exploratory_leadlag',
        ))

# ── Registry laden & speichern ───────────────────────────────

if os.path.exists(REGISTRY_FILE):
    with open(REGISTRY_FILE, encoding='utf-8') as f:
        registry = json.load(f)
else:
    registry = {}

current_candidate_ids = {c['id'] for c in factory_candidates}
new_count = 0

# Phase 1: aktuelle Kandidaten einpflegen, aktivieren
for c in factory_candidates:
    hid = c['id']
    if hid in registry:
        c['review_status']          = registry[hid].get('review_status', 'unreviewed')
        c['include_in_report']      = registry[hid].get('include_in_report', False)
        c['include_in_model_logic'] = registry[hid].get('include_in_model_logic', False)
        c['reviewer_notes']         = registry[hid].get('reviewer_notes', '')
        registry[hid]['last_updated']          = RUN_TS
        registry[hid]['active_in_current_run'] = True
        registry[hid]['last_seen_run']         = RUN_TS
        registry[hid]['missing_runs']          = 0
    else:
        c['reviewer_notes'] = ''
        registry[hid] = {
            'review_status':          'unreviewed',
            'include_in_report':      False,
            'include_in_model_logic': False,
            'reviewer_notes':         '',
            'first_seen':             RUN_TS,
            'last_updated':           RUN_TS,
            'last_seen_run':          RUN_TS,
            'active_in_current_run':  True,
            'missing_runs':           0,
            'lifecycle_status':       'active',
        }
        new_count += 1

    c['promotion_blocked_reason'] = promotion_blocked_reason(
        c['n'], c['pattern_status'], c['review_status']
    )

    fp = os.path.join(CANDIDATES_DIR, f'{hid}.json')
    with open(fp, 'w', encoding='utf-8') as f:
        json.dump(c, f, indent=2, ensure_ascii=False)

# Phase 2: Lifecycle aktualisieren.
# WICHTIG (Lifecycle-Trennung): candidate_status sagt nur, ob die Factory den
# Kandidaten IM AKTUELLEN LAUF erzeugt hat. Das ist KEINE fachliche Aussage:
# "fehlt im Factory-Output" != "widerlegt". Auto-Retire per missing_runs gilt
# deshalb NUR fuer unreviewte Kandidaten. Reviewte Eintraege (review_status !=
# 'unreviewed') behalten einen review-eigenen Lifecycle und werden NIE
# automatisch pensioniert — nur der Reviewer selbst kann sie zuruecknehmen.
dormant_count = 0
retired_count = 0
for hid, meta in registry.items():
    present = hid in current_candidate_ids
    meta['candidate_status'] = 'active' if present else 'absent_this_run'
    if not present:
        meta['active_in_current_run'] = False
        meta['missing_runs'] = meta.get('missing_runs', 0) + 1   # rein informativ
    rs = meta.get('review_status', 'unreviewed')
    if rs != 'unreviewed':
        # Review-eigener Lifecycle — unabhaengig von Factory-Praesenz
        if rs == 'reviewed' and meta.get('include_in_model_logic'):
            meta['lifecycle_status'] = 'accepted'       # vom Reviewer ins Modell uebernommen
        else:
            meta['lifecycle_status'] = 'reviewed_held'  # fachlich entschieden, beobachtend gehalten
        continue
    # Unreviewte Kandidaten: absenz-basierter Lifecycle wie bisher
    if present:
        meta['lifecycle_status'] = 'active'
    elif meta['missing_runs'] >= 10:
        meta['lifecycle_status'] = 'retired'
        retired_count += 1
    elif meta['missing_runs'] >= 5:
        meta['lifecycle_status'] = 'dormant'
        dormant_count += 1
    else:
        meta['lifecycle_status'] = 'idle'

with open(REGISTRY_FILE, 'w', encoding='utf-8') as f:
    json.dump(registry, f, indent=2, ensure_ascii=False)

# ── Ausgabe ──────────────────────────────────────────────────

categories = ['core', 'diagnostic', 'exploratory_leadlag']
cat_labels  = {
    'core':               '── CORE CANDIDATES ──',
    'diagnostic':         '── DIAGNOSTIC CANDIDATES ──',
    'exploratory_leadlag':'── EXPLORATORY LEAD-LAG POOL ──',
}
ev_icon = {
    'exploratory_signal': '🔬',
    'testable':           '🧪',
    'moderate_evidence':  '📊',
    'robust_candidate':   '✔️',
    'unknown':            '❓',
}
pat_icon = {'detected': '✅', 'unclear': '⚠️', 'absent': '❌'}

print('HYPOTHESIS FACTORY v2')
print('=' * 78)
print(f'  Kandidaten gesamt: {len(factory_candidates)}  |  Neu: {new_count}  |  Registry: {REGISTRY_FILE}')
print(f'  Lifecycle: dormant={dormant_count}  retired={retired_count}  (in Registry: {len(registry)})')

for cat in categories:
    group = [c for c in factory_candidates if c['category'] == cat]
    if not group: continue
    print(f'\n  {cat_labels[cat]}')
    for c in sorted(group, key=lambda x: {'high':0,'medium':1,'low':2,'quarantine':3}[x['priority']]):
        pat  = pat_icon.get(c['pattern_status'], '?')
        ev   = ev_icon.get(c['evidence_level'], '?')
        n_str = f'n={c["n"]}' if c['n'] is not None else 'n=?'
        r_str = f'r={c["pearson"]:+.3f}' if c.get('pearson') is not None else ''
        blocked = f'  🔒 {c["promotion_blocked_reason"]}' if c['promotion_blocked_reason'] else ''
        print(f'  {pat} {ev} [{c["priority"]:<6}] {c["id"]:<40} {n_str:<6} {r_str}{blocked}')
        if c['mechanism']:
            print(f'      └─ {c["mechanism"][:75]}')

print(f'\n  Legende: ✅=detected ⚠️=unclear  🔬=exploratory 🧪=testable 📊=moderate ✔️=robust')



## 14. Export

`layer8_state.json` — maschinenlesbar für Layer 9 oder spätere Auswertung.
`layer8_report.md` — lesbarer Bericht.

In [ ]:
# ============================================================
# EXPORT — layer8_state.json
# ============================================================

layer8_state = {
    'timestamp':      RUN_TS,
    'engine_version': ENGINE_VERSION,
    'layer':          8,
    'n_snapshots':    len(snaps),
    'n_pairs':        n_pairs,

    # Kern-Analysen
    'state_profile':          state_freq,
    'layer_stats':            layer_stats,
    'dl3_analysis': {
        'data':           dl3_data,
        'threshold':      dl3_threshold,
        'anomal_mean':    round(float(np.mean(anomal_dl3)), 4) if anomal_dl3 else None,
        'seasonal_mean':  round(float(np.mean(seasonal_dl3)), 4) if seasonal_dl3 else None,
    },
    'combined_activation': {
        'data':               combined_data,
        'threshold':          combined_threshold,
        'overlap_combined':   overlap_combined if combined_threshold else None,
        'overlap_dl3':        overlap_dl3 if combined_threshold else None,
    },
    'carnegie_amplitude':     carnegie_amplitude,
    'l2_l3_paradox':          l2_l3_paradox,
    'coupling_summary':       coupling_summary,
    'operator_analysis':      operator_analysis,

    # Neu
    'enso_coupling_analysis': enso_coupling_analysis,
    'cavity_events':          cavity_events,

    # Hypothesen
    'hypothesis_candidates':  factory_candidates,

    # Makro-Layer Handoff
    'macro_handoff': {
        'atmosphere_state_vector': {
            'dominant_state':     max(state_freq, key=lambda s: state_freq[s]['count']),
            'layer_means':        {l: layer_stats[l]['mean'] for l in layer_stats},
            'downstream_score':   round(float(np.mean([
                layer_stats.get('L3_atmosphere', {}).get('mean', 0),
                layer_stats.get('L5_global_electric_circuit', {}).get('mean', 0),
                layer_stats.get('L6_resonance_field', {}).get('mean', 0),
            ])), 4),
            'preparation_score':  layer_stats.get('L2_surface_zone', {}).get('mean', 0),
            'external_score':     layer_stats.get('L0_external_drivers', {}).get('mean', 0),
        },
        'atmosphere_regime':       enso_coupling_analysis['macro_handoff'],
        'field_activation': {
            'operator_vector':     operator_analysis.get('latest_ranking', []),
            'coverage_regime':     operator_analysis.get('coverage_regime', 'unknown'),
        },
        'atmospheric_anomalies': {
            'persistent_baseline_tags': persistent if persistent else [],
            'dl3_threshold':            dl3_threshold,
            'combined_threshold':       combined_threshold,
        },
        'confidence':              evidence_level(len(snaps)),
        'ready_for_macro_layer':   len(snaps) >= 20,
        'n_snapshots':             len(snaps),
    },
}

# numpy-Bereinigung
def _to_python(obj):
    if isinstance(obj, dict):     return {k: _to_python(v) for k, v in obj.items()}
    if isinstance(obj, list):     return [_to_python(v) for v in obj]
    if isinstance(obj, np.bool_):    return bool(obj)
    if isinstance(obj, np.integer):  return int(obj)
    if isinstance(obj, np.floating): return None if np.isnan(obj) else float(obj)
    return obj
layer8_state = _to_python(layer8_state)
from atmosphere.meta.role_proxy_writeback import annotate_l8
layer8_state = annotate_l8(layer8_state)
with open(STATE_FILE, 'w', encoding='utf-8') as f:
    json.dump(layer8_state, f, indent=2, ensure_ascii=False)
print(f'✅ {STATE_FILE} gespeichert')

# ── Zusammenfassung ──────────────────────────────────────────
mh = layer8_state['macro_handoff']
print('\n' + '=' * 78)
print('LAYER 8 — MACRO HANDOFF SUMMARY')
print('=' * 78)
print(f'  Snapshots:            {len(snaps)}  ({evidence_level(len(snaps))})')
print(f'  Tagespaare:           {n_pairs}')
print(f'  Dominant State:       {mh["atmosphere_state_vector"]["dominant_state"]}')
print(f'  Downstream-Score:     {mh["atmosphere_state_vector"]["downstream_score"]:.3f}')
print(f'  Preparation-Score:    {mh["atmosphere_state_vector"]["preparation_score"]:.3f}  (L2)')
print(f'  ENSO Macro-Phase:     {mh["atmosphere_regime"]["enso_macro_phase"]}')
print(f'  ENSO Event-Risk:      {mh["atmosphere_regime"].get("enso_event_risk") or mh["atmosphere_regime"].get("event_risk", "unknown")}')
print(f'  Feedback-Risk:        {mh["atmosphere_regime"]["feedback_risk"]}')
print(f'  Operator Coverage:    {mh["field_activation"]["coverage_regime"]}')
print(f'  Ready for Macro:      {mh["ready_for_macro_layer"]}  (>= 20 Snapshots)')
print('=' * 78)

In [ ]:
# Markdown-Report
lines = []
lines.append(f'# Layer 8 — Research Report')
lines.append(f'')
lines.append(f'**Run:** {RUN_TS}')
lines.append(f'**Snapshots analysiert:** {len(snaps)}')
lines.append(f'**Vollständige Tagespaare:** {n_pairs}')
lines.append(f'')
lines.append(f'## System-State Häufigkeit')
lines.append(f'')
for state, st in state_freq.items():
    lines.append(f'- `{state}` — {st["count"]}× ({st["pct"]}%)')
lines.append(f'')
lines.append(f'## Tagespaare (ΔL3 Aktivierung)')
lines.append(f'')
lines.append(f'| Datum | L3 früh | L3 abend | ΔL3 | Abend-State |')
lines.append(f'|---|---|---|---|---|')
for r in dl3_data:
    state_short = r['evening_state'].replace('_state','')
    lines.append(f'| {r["date"]} | {r["L3_morning"]:.3f} | {r["L3_evening"]:.3f} | {r["delta_L3"]:+.3f} | {state_short} |')
if dl3_threshold is not None:
    lines.append(f'')
    lines.append(f'**ΔL3-Schwelle (empirisch):** {dl3_threshold:+.3f}')
lines.append(f'')
lines.append(f'## L2 ↔ L3 Beziehung')
lines.append(f'')
lines.append(f'- Pearson: **{pearson_l2_l3:+.3f}**')
lines.append(f'- L2 Trend: {l2_trend:+.5f} / Snapshot')
lines.append(f'- L3 Trend: {l3_trend:+.5f} / Snapshot')
lines.append(f'- Gap-Trend: {gap_trend:+.5f} / Snapshot')
lines.append(f'- {l2_l3_paradox["interpretation"]}')
lines.append(f'')
# ── Confound-aware rendering (respektiert den L8 Promotion-Guard) ───────────
from atmosphere.meta.role_proxy_writeback import OPERATOR_COMPOSITION, classify_independence

def _op_circular_vs_dl3(op_name):
    short = op_name.replace('_operator', '')
    return 'L3_atmosphere' in OPERATOR_COMPOSITION.get(short, [])

# factory_candidates wurde von annotate_l8() in-place getaggt (Zelle 32)
# WICHTIG: aus der von annotate_l8 getaggten Liste lesen, NICHT aus factory_candidates.
# _to_python() in Zelle 32 macht eine Deep-Copy -> factory_candidates bleibt untagged.
_cand_indep = {c['id']: c.get('evidence_independence') for c in layer8_state['hypothesis_candidates']}
def _hyp_confound(h):
    hid = h['id']
    if hid in _cand_indep and _cand_indep[hid]:
        return _cand_indep[hid]
    if hid.startswith('H_op_'):
        return classify_independence(hid[len('H_op_'):] + '_operator')[0]
    if hid.startswith('H_leadlag_'):
        return classify_independence(hid[len('H_leadlag_'):])[0]
    return None

# dedup hypotheses by id (strongest status wins) -> behebt die H1-Dopplung
_RANK = {'confirmed': 4, 'likely': 3, 'mixed': 2, 'open': 1}
_seen = {}
for _h in hypotheses:
    _hid = _h['id']
    if _hid not in _seen or _RANK.get(_h.get('status'), 0) > _RANK.get(_seen[_hid].get('status'), 0):
        _seen[_hid] = _h
hypotheses_report = list(_seen.values())

lines.append(f'## Field Operators')
lines.append(f'')
if operator_analysis.get('status') == 'active':
    lines.append(f'Coverage: {operator_analysis["snapshots_with_operators"]}/{operator_analysis["total_snapshots"]} Snapshots')
    lines.append(f'')
    if operator_analysis.get('latest_ranking'):
        lines.append(f'### Aktuelle Operator-Rangliste')
        lines.append(f'')
        for r in operator_analysis['latest_ranking']:
            short = r['operator'].replace('_operator', '')
            flag = '  ⚠️ confounded_circular' if _op_circular_vs_dl3(r['operator']) else ''
            lines.append(f'- **{short}**: {r["score"]:.3f} — {r["interpretation"]}{flag}')
    if operator_analysis.get('operator_dl3_correlation'):
        lines.append(f'')
        lines.append(f'### Operator ↔ ΔL3 Korrelation')
        lines.append(f'')
        for op, r in sorted(operator_analysis['operator_dl3_correlation'].items(), key=lambda x: -abs(x[1])):
            short = op.replace('_operator', '')
            flag = '  ⚠️ **confounded_circular** — Operator enthält L3, kein unabhängiger Prädiktor' if _op_circular_vs_dl3(op) else ''
            lines.append(f'- {short}: r = {r:+.3f}{flag}')
else:
    lines.append(f'_{operator_analysis.get("status", "unknown")}_ — mehr Snapshots benötigt.')
lines.append(f'')
lines.append(f'## Hypothesen')
lines.append(f'')
status_emoji = {'confirmed': '✅', 'likely': '🔶', 'open': '❓', 'mixed': '⚠️'}
for h in hypotheses_report:
    cf = _hyp_confound(h)
    if cf and cf != 'independent':
        lines.append(f'### 🚫 {h["id"]}: {h["question"]}')
        lines.append(f'')
        lines.append(f'- **Status:** confound_blocked ({cf}) — nicht promotbar, nur exploratorisch')
        lines.append(f'- **Evidenz:** {h["evidence"]}')
        lines.append(f'- **Nächster Schritt:** {h["next_step"]}')
        lines.append(f'')
    else:
        em = status_emoji.get(h['status'], '•')
        tag = '  ✓ unabhängig (Backbone)' if cf == 'independent' else ''
        lines.append(f'### {em} {h["id"]}: {h["question"]}{tag}')
        lines.append(f'')
        lines.append(f'- **Status:** {h["status"]}')
        lines.append(f'- **Evidenz:** {h["evidence"]}')
        lines.append(f'- **Nächster Schritt:** {h["next_step"]}')
        lines.append(f'')

with open(REPORT_FILE, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print(f'✅ {REPORT_FILE} gespeichert')